In [1]:
import os
from getpass import getpass

if not os.path.exists("/content/riskml-capstone"):
    token = getpass("GitHub Personal Access Token: ")
    !git clone https://{token}@github.com/stevearchuleta/riskml-capstone.git /content/riskml-capstone

os.chdir("/content/riskml-capstone")
!pwd

GitHub Personal Access Token: ··········
Cloning into '/content/riskml-capstone'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 257 (delta 101), reused 181 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 11.02 MiB | 27.65 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/riskml-capstone


# Notebook 04:
# Risk Forecasting — DAG-Constrained Pipeline, Prefix-Based Feature Gating, and Causal vs Baseline Comparison

---
<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px; margin-bottom: 15px;">
<strong>📘 PURPOSE:</strong> A reproducible modeling notebook that:

<div style="margin-left: 24px; margin-top: 8px;">

**(1)** implements prefix-based feature gating to restrict the Risk forecast stage to DAG-permitted parents only (VOL, MACRO, REGIME),

**(2)** trains a DAG-constrained XGBoost model using the identical hyperparameters and expanding-window walk-forward protocol established in Notebook 03,

**(3)** computes per-ticker and aggregate RMSE and MAE deltas against the unconstrained EWMA and XGBoost baselines,

**(4)** extracts constrained feature importance to confirm that no DAG-excluded features (MOM, VAL, ML, SENT) appear, and

**(5)** computes factor-exposure entropy diagnostics to evaluate whether the constraint layer produces more balanced feature loadings.

</div>

Notebook 04 is the primary experiment — testing <strong>Hypothesis H1 (forecast accuracy)</strong> and generates the interpretability evidence for <strong>Hypothesis H4 (factor-exposure entropy)</strong>.
</div>


## CAPSTONE CONTEXT

<div style="border-left: 4px solid #6a1b9a; padding-left: 12px; margin: 10px 0;">

**Title:**  
<span style="color: purple;"><strong>Causal-Aware, Machine-Learning-Driven Risk Forecasting and Factor Construction:</strong></span> A Python–Azure Pipeline Integrating NLP, Directed Factor Constraints, and Portfolio Analytics

**Thesis:**  
A small, theory-driven manually constrained <span style="color: purple;"><strong>Directed Acyclic Graph (DAG)</strong></span> that restricts information flow can improve the stability and interpretability of ML-based risk forecasting and factor-based portfolio allocation, relative to unconstrained baselines, under regime variation and estimation noise.

**Research Question:**  
How does imposing <span style="color: purple;"><strong>manual causal constraints</strong></span> on an ML-driven risk forecasting pipeline affect forecast accuracy, portfolio performance, and interpretability compared to unconstrained baselines?

</div>

---

## MANUAL DAG (Conceptual Constraint Layer)

<div style="border: 2px solid #6a1b9a; background-color: #f3e5f5; padding: 12px; border-radius: 5px;">
<strong>🔗 DAG STRUCTURE:</strong>

```
Sentiment  →  Momentum  →  Returns
```
<span style="color: purple;">Path 1: Sentiment <strong>AMPLIFIES</strong> Momentum; Momentum <strong>FORECASTS</strong> Returns</span>

```
Value (HML)            →  Returns
```
<span style="color: purple;">Path 2: Value (HML) <strong>EXPLAINS</strong> Returns</span>

```
Volatility →  Risk     →  Allocation
```
<span style="color: purple;">Path 3: Volatility <strong>ESTIMATES</strong> Risk; Risk <strong>CONSTRAINS</strong> Allocation</span>

<span style="color: purple;"><strong>Notebook 04 Role in the DAG Pipeline:</strong></span> Notebook 03 built <strong>unconstrained</strong> baseline models that accessed <strong>all</strong> 151 non-sentiment features regardless of DAG-node membership. Notebook 04 now <strong>enforces</strong> the DAG constraint layer — restricting the Risk forecast stage to features from the <strong>Volatility</strong> node (plus exogenous MACRO and REGIME conditioning variables) and excluding Momentum, Value, ML latent, and Sentiment features entirely. The constrained model receives <strong>74 features</strong> versus 151 in the unconstrained baseline, a 51% feature reduction that serves as implicit regularization. By comparing constrained-model performance against the Notebook 03 baselines, Notebook 04 isolates the effect of DAG-based information-flow restriction on forecast accuracy and feature-importance interpretability.
</div>

---

## DAG CONSTRAINT IMPLEMENTATION — PREFIX-BASED FEATURE GATING

<div style="border: 2px solid #6a1b9a; background-color: #f3e5f5; padding: 12px; border-radius: 5px;">
<strong>🔗 ALLOWED-PARENT SET FOR THE RISK FORECAST STAGE:</strong>

The Risk node in the DAG receives information only from the Volatility node. MACRO and REGIME serve as exogenous conditioning variables available to all nodes. The feature-gating logic parses the double-underscore prefix of each column name and admits only columns matching the allowed-parent set.

| Status | Prefix | Feature Count | DAG Justification |
|:-------|:-------|:--------------|:------------------|
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `VOL__` | 70 | Volatility → Risk (direct parent edge) |
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `MACRO__` | 3 | Exogenous conditioning (VIX, T10Y2Y, DTB3) |
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `REGIME__` | 1 | Exogenous conditioning (VIX high/low regime indicator) |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `MOM__` | 56 | Momentum → Returns, not Risk |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `VAL__` | 18 | Value → Returns, not Risk |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `ML__` | 3 | PCA latent factors not assigned as Risk parents |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `SENT__` | 1 | 100% NaN placeholder; Sentiment → Momentum, not Risk |
| | **TOTAL ALLOWED** | **74** | |
| | **TOTAL EXCLUDED** | **77 non-NaN + 1 NaN** | |

<span style="color: purple;"><strong>Forbidden Shortcuts:</strong></span> The DAG forbids the following information-flow paths into the Risk node: Sentiment → Risk (sentiment must flow through Momentum first), Momentum → Risk (momentum affects Returns, not Risk directly), Value → Risk (value affects Returns, not Risk directly), ML latent factors → Risk (PCA scores are general-purpose and the DAG does not assign ML factors as Risk parents). Only the Volatility → Risk edge is permitted, plus exogenous MACRO/REGIME conditioning.
</div>

---

## HYPOTHESES TESTED IN NOTEBOOK 04

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 HYPOTHESIS H1 — FORECAST ACCURACY:</strong>

The DAG-constrained pipeline reduces out-of-sample MAE and RMSE on 20-trading-day forward realized volatility relative to the unconstrained XGBoost baseline from Notebook 03.

**Test mechanism:** Compare per-ticker and aggregate RMSE and MAE between the constrained model (74 features) and the unconstrained XGBoost baseline (151 features), using identical hyperparameters and the identical expanding-window walk-forward protocol.
</div>

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📐 HYPOTHESIS H4 — INTERPRETABILITY (FACTOR-EXPOSURE ENTROPY):</strong>

Factor-exposure entropy increases under DAG constraints, indicating more balanced feature loadings and reduced concentration on a few cross-node shortcuts.

**Test mechanism:** Compute Shannon entropy $H = -\sum_k p_k \ln(p_k)$ over normalized gain shares from the constrained versus unconstrained XGBoost feature-importance vectors. Higher entropy indicates more dispersed feature usage.

$$
H = -\sum_{k=1}^{K} p_k \ln(p_k), \quad p_k = \frac{\text{gain}_k}{\sum_{j=1}^{K} \text{gain}_j}
$$
</div>

---

## FROZEN HYPERPARAMETERS (FROM NOTEBOOK 03 TUNING)

<div style="border: 2px solid #ff9933; background-color: #fff3cd; padding: 12px; border-radius: 5px;">
<strong>⚠️ CRITICAL FAIRNESS CONSTRAINT:</strong> Notebook 04 reuses the <strong>exact</strong> hyperparameters selected during Notebook 03 walk-forward validation. Freezing hyperparameters ensures that any performance difference between the constrained and unconstrained models is attributable to the feature restriction, not to hyperparameter variation.

| Parameter | Frozen Value |
|:----------|:-------------|
| `n_estimators` | 400 |
| `max_depth` | 4 |
| `learning_rate` | 0.05 |
| `subsample` | 0.80 |
| `colsample_bytree` | 0.80 |
| `reg_lambda` | 1.0 |
| `objective` | `reg:squarederror` |
| `random_state` | 692 |
| `tree_method` | `hist` |
</div>

---

## EVALUATION PROTOCOL (REUSED FROM NOTEBOOK 03)

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 DEFINITION — Expanding-Window Walk-Forward Evaluation:</strong>

Notebook 04 reuses the identical evaluation protocol from Notebook 03 to ensure a fair comparison:

| Dimension | Value |
|:----------|:------|
| **Train–Validation–Test split** | 60%–20%–20% of the effective modeling window (time-ordered, no shuffling) |
| **Forecast horizon** | 20 trading days |
| **Walk-forward step size** | 20 trading days |
| **Label-safe boundary** | Training data truncated at `block_start − 20` to prevent label leakage from overlapping forward windows |
| **Evaluation metrics** | RMSE (primary) and MAE (robust complement), per-ticker and aggregate equal-weight mean |
| **Baseline references** | EWMA (span-63) and unconstrained XGBoost (151 features), both from Notebook 03 saved artifacts |

<span style="color: darkblue;"><strong>Walk-forward protocol:</strong></span> At each evaluation step, the training window expands by 20 trading days. A fresh XGBoost model is fit on the expanded training window and produces a forecast for the next 20-day block. The expanding-window design mimics the information set available to a portfolio manager making monthly allocation decisions.
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING — TEMPORAL INTEGRITY:</strong> The walk-forward evaluation must never allow information from future dates to enter the training set. At each evaluation step, the model accesses only features and targets from dates strictly before the block start minus the 20-day forecast horizon. The label-safe boundary ensures that no partially overlapping forward-volatility targets contaminate the training window. The TARGET__-prefixed columns are loaded from a <strong>separate file</strong> (<code>target_fwd_vol.parquet</code>) and must never appear in the feature matrix.
</div>

---

## NOTEBOOK 03 BASELINE BENCHMARKS (REFERENCE)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 BASELINE PERFORMANCE (from Notebook 03 saved artifacts):</strong>

Notebook 04 loads the following Notebook 03 artifacts for direct comparison:

| Artifact | Path | Contents |
|:---------|:-----|:---------|
| Baseline metrics | `reports/tables/notebook03_baseline_rmse_mae.csv` | Per-ticker RMSE and MAE for EWMA and XGBoost |
| Baseline predictions | `reports/tables/notebook03_baseline_predictions_long.csv` | Long-form predictions for all 14 tickers and all test blocks |
| Feature importance | `reports/tables/notebook03_xgb_feature_importance.csv` | Gain-based feature importance from the unconstrained SPY model |
| Tuning results | `reports/tables/notebook03_xgb_tuning_results.csv` | 6-candidate tuning grid with validation RMSE |

**Aggregate baseline metrics (equal-weight mean across 14 tickers):**

| Model | RMSE | MAE |
|:------|:-----|:----|
| EWMA (span-63) | 0.0708 | 0.0483 |
| XGBoost (unconstrained, 151 features) | 0.0698 | 0.0491 |

<span style="color: darkorange;"><strong>⚠ Key finding from Notebook 03:</strong></span> A momentum feature (`MOM__XLF__cum_ret__10d`) appeared at rank 2 in the unconstrained SPY feature-importance hierarchy — a cross-node shortcut from Momentum into the Risk forecast. Under DAG constraints, momentum features are excluded from the Risk node's allowed-parent set. Whether the constrained model maintains or improves RMSE without momentum cross-contamination is the central empirical question of Notebook 04.
</div>

---

## INPUT AND OUTPUT FILES

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px;">
<strong>📁 INPUT (from Notebooks 02 and 03):</strong>

- `data/processed/features.parquet` — Feature matrix: 2,798 rows × 152 columns (DAG-node-prefixed)
- `data/processed/target_fwd_vol.parquet` — Forward realized volatility target: 2,798 rows × 14 columns
- `data/processed/etf_returns.parquet` — Daily log returns for 14 ETFs
- `reports/tables/notebook03_baseline_rmse_mae.csv` — EWMA and XGBoost per-ticker metrics
- `reports/tables/notebook03_baseline_predictions_long.csv` — Long-form baseline predictions (all tickers, all test blocks)
- `reports/tables/notebook03_xgb_feature_importance.csv` — Unconstrained feature importance (SPY)
- `reports/tables/notebook03_xgb_tuning_results.csv` — Hyperparameter tuning grid

</div>

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📁 OUTPUT (produced by Notebook 04):</strong>

**Tables (`reports/tables/`):**
- `notebook04_causal_rmse_mae.csv` — Per-ticker RMSE and MAE for the constrained model
- `notebook04_causal_predictions_long.csv` — Long-form constrained predictions (all tickers, all test blocks)
- `notebook04_dag_gating_summary.csv` — Feature-level allowed/excluded status with DAG prefix
- `notebook04_effective_window_summary.csv` — Effective modeling window confirmation
- `notebook04_split_summary.csv` — Train–validation–test split boundaries
- `causal_vs_baseline_rmse_mae.csv` — Merged comparison table with per-ticker RMSE/MAE deltas
- `notebook04_causal_feature_importance.csv` — Gain-based feature importance for representative tickers
- `notebook04_entropy_comparison.csv` — Constrained vs unconstrained entropy diagnostics
- `notebook04_prefix_gain_share.csv` — DAG-prefix-level gain share comparison
- `notebook04_top_feature_comparison.csv` — Top-10 features for constrained vs unconstrained
- `notebook04_asset_class_comparison.csv` — Asset-class-level RMSE/MAE delta summary
- `notebook04_artifact_manifest.csv` — Registry of all Notebook 04 output artifacts

**Figures (`reports/figures/`):**
- `notebook04_dag_figure.png` — Manual DAG architecture diagram with node boxes and directed edges
- `notebook04_forecast_vs_realized_SPY.png` — Forecast overlay: EWMA vs XGBoost vs Causal XGBoost vs realized
- `notebook04_causal_vs_baseline_rmse_delta.png` — RMSE delta bar chart (constrained minus baselines, per ticker)
- `notebook04_entropy_comparison.png` — Normalized entropy bar chart (constrained vs unconstrained)
- `notebook04_causal_feature_importance_SPY.png` — Top constrained features by gain (SPY)
- `notebook04_prefix_gain_share_SPY.png` — DAG-prefix gain share comparison (SPY)

**Data (`data/processed/`):**
- `notebook04_causal_predictions_long.csv` — Predictions copy for downstream Notebook 05 and Notebook 06 consumption

**Models (`reports/models/`):**
- `notebook04_xgb_causal_{TICKER}.json` — Serialized constrained XGBoost models for representative tickers

</div>

---

## FEATURE NAMING CONVENTION (REFERENCE)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 NAMING PATTERNS (inherited from Notebook 02):</strong>

**Asset-level features** (one column per ETF per window):
<code>{DAG_NODE}__{TICKER}__{feature_name}__{window}</code>

**Market-level features** (one column shared across all ETFs):
<code>{DAG_NODE}__{feature_name}__{window(optional)}</code>

**Examples:**
- `VOL__SPY__rvol__21d` — Volatility node, SPY, 21-day realized volatility (**ALLOWED** for Risk stage)
- `VOL__SPY__ewma_vol__span63` — Volatility node, SPY, EWMA volatility with span 63 (**ALLOWED** for Risk stage)
- `MACRO__vixcls` — Macro conditioning, VIX closing level (**ALLOWED** for Risk stage)
- `REGIME__vix_high` — Regime conditioning, binary high-VIX indicator (**ALLOWED** for Risk stage)
- `MOM__SPY__cum_ret__21d` — Momentum node, SPY, 21-day cumulative return (**EXCLUDED** from Risk stage)
- `VAL__SPY__hml_beta__63d` — Value node, SPY, 63-day rolling beta to HML (**EXCLUDED** from Risk stage)
- `ML__pc1` — ML latent factor, first principal component (**EXCLUDED** from Risk stage)
- `SENT__sentiment_market` — Sentiment node, daily market sentiment score (**EXCLUDED**; 100% NaN placeholder)

The double-underscore delimiter enables programmatic DAG-node parsing. The first token is always the DAG-node prefix. Notebook 04 uses the prefix to enforce causal constraints via the `gate_features_by_prefix()` function.
</div>

---

## REPRODUCIBILITY

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ REPRODUCIBILITY GUARANTEES:</strong>

| Dimension | Policy |
|:----------|:-------|
| **Random Seed** | 692 (set in setup cell; passed to XGBoost `random_state`, NumPy, and Python `random`) |
| **Time Ordering** | Strictly preserved — expanding-window walk-forward with no shuffling |
| **Look-Ahead Leakage** | At each evaluation step, the model accesses only features and targets from dates before the label-safe boundary |
| **Hyperparameter Fairness** | Frozen from Notebook 03 tuning — no re-tuning on constrained features |
| **DAG Constraint Proof** | Explicit gating validation cell prints allowed and excluded feature lists with prefix counts |
| **Effective Modeling Mask** | Reuses the identical completeness mask from Notebook 03 (all features + target simultaneously non-NaN) |
| **Sentiment Exclusion** | `SENT__sentiment_market` column (100% NaN) excluded before feature gating |
| **Target-Column Governance** | All target access routes through `TARGET_COL_MAP` dictionary — bare ticker access is forbidden |
| **Model Serialization** | Constrained XGBoost models saved to JSON for reproducibility and downstream comparison |
| **Artifact Persistence** | All tables, figures, models, and predictions saved to `reports/` and `data/processed/` |
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING:</strong> Notebook 04 trains DAG-constrained models and evaluates forecast accuracy against Notebook 03 baselines. Three leakage boundaries must hold simultaneously: (1) <strong>Temporal leakage:</strong> the expanding-window protocol with label-safe boundary prevents future target values from entering the training set. (2) <strong>Feature leakage:</strong> TARGET__-prefixed columns are loaded from a separate file (<code>target_fwd_vol.parquet</code>) and must never appear in the feature matrix. (3) <strong>DAG leakage:</strong> the prefix-gating function must exclude all MOM__, VAL__, ML__, and SENT__ columns from the constrained model's input — the gating validation cell confirms that no forbidden-prefix columns survived the filter.
</div>

---

## NOTEBOOK STRUCTURE OVERVIEW

| Cell Block | Code Block | Purpose | Key Actions |
|:-----------|:-----------|:--------|:------------|
| **SETUP** | CODE_BLOCK_A | Environment initialization | Imports, paths, seed, display options, helper functions, Notebook 03 artifact paths |
| **LOAD** | CODE_BLOCK_B | Load features, targets, returns, and Notebook 03 artifacts | Read Parquet files, load optional NB03 CSVs, validate index alignment, drop sentiment |
| **EFFECTIVE WINDOW** | CODE_BLOCK_C | Apply effective modeling mask | Completeness intersection, target-column governance map, effective window summary |
| **DAG GATING** | CODE_BLOCK_D | Define and apply DAG constraint layer | Manual DAG edge list, prefix gating function, allowed/excluded validation, gating summary |
| **SPLIT** | CODE_BLOCK_E | Time-ordered train–validation–test split | 60/20/20 split, expanding-window block indices, label-safe boundary function |
| **CAUSAL MODEL** | CODE_BLOCK_F | Train and evaluate DAG-constrained XGBoost | Frozen hyperparameters, walk-forward prediction loop, per-ticker RMSE/MAE, prediction export |
| **COMPARISON** | CODE_BLOCK_G | Merge constrained metrics with Notebook 03 baselines | RMSE/MAE delta computation, win counts versus XGBoost and EWMA baselines |
| **IMPORTANCE** | CODE_BLOCK_H | Extract constrained feature importance | Gain-based importance for representative tickers, forbidden-prefix purity check |
| **ENTROPY** | CODE_BLOCK_I | Compute factor-exposure entropy diagnostics | Shannon entropy, normalized entropy, concentration statistics, constrained vs unconstrained |
| **PLOT DATA** | CODE_BLOCK_J | Prepare data for final figures | Combined prediction table, prefix gain shares, top-feature comparison table |
| **PLOT PREP** | CODE_BLOCK_K | Build pivot tables and delta tables for plotting | Forecast pivot, RMSE/MAE delta tables, aggregate summary |
| **ASSET CLASS** | CODE_BLOCK_L | Asset-class-level decomposition | Map tickers to asset classes, compute mean RMSE/MAE deltas by asset class |
| **REGISTRY** | CODE_BLOCK_M | Preliminary artifact registry | List all expected output paths, check existence flags before figure creation |
| **FIGURES** | CODE_BLOCK_N | Generate all Notebook 04 figures and final manifest | DAG diagram, forecast overlay, RMSE delta bars, entropy bars, importance bars, prefix gain bars, artifact manifest |

---

*Notebook 04 of 7 | MScFE 692 Capstone | Steven Archuleta | WorldQuant University*


In [13]:
# ============
# CODE_BLOCK_A
# ============

# =================================================================
# IMPORT WARNINGS FOR CLEAN NOTEBOOK OUTPUT VIA WARNING SUPPRESSION
# =================================================================

import warnings

# ============================================================
# SUPPRESS NON-CRITICAL WARNINGS TO KEEP AUDIT OUTPUT READABLE
# ============================================================

warnings.filterwarnings("ignore")

# =================================================================
# IMPORT RANDOM FOR REPRODUCIBLE PYTHON-LEVEL STOCHASTIC OPERATIONS
# =================================================================

import random

# ======================================================================
# IMPORT JSON FOR ARTIFACT SERIALIZATION AND HUMAN-READABLE PATH LOGGING
# ======================================================================

import json

# ===================================================================
# IMPORT OS FOR ENVIRONMENT INSPECTION AND OPTIONAL COLAB PATH CHECKS
# ===================================================================

import os

# ==========================================
# IMPORT SYS FOR PYTHON VERSION AUDIT OUTPUT
# ==========================================

import sys

# ==========================================================
# IMPORT MATH FOR SAFE LOG OPERATIONS IN ENTROPY CALCULATION
# ==========================================================

import math

# ============================================================
# IMPORT PATH FOR CROSS-PLATFORM FILE AND DIRECTORY MANAGEMENT
# ============================================================

from pathlib import Path

# ==========================================================================
# IMPORT OPTIONAL AND COLLECTION TYPE HINTS FOR READABLE FUNCTION SIGNATURES
# ==========================================================================

from typing import Dict, List, Optional, Tuple

# =============================================================
# IMPORT NUMPY FOR NUMERICAL COMPUTATION AND SQRT ANNUALIZATION
# =============================================================

import numpy as np

# ======================================================================
# IMPORT PANDAS FOR DATAFRAME TRANSFORMS AND PARQUET OR CSV INPUT OUTPUT
# ======================================================================

import pandas as pd

# =================================================================
# IMPORT MATPLOTLIB FOR REPORT-READY FIGURES SAVED AS PNG ARTIFACTS
# =================================================================

import matplotlib.pyplot as plt

# ===========================================================================
# IMPORT MATPLOTLIB PATCHES FOR DAG NODE BOX DRAWING IN THE FINAL FIGURE CELL
# ===========================================================================

from matplotlib.patches import FancyBboxPatch

# ====================================================
# IMPORT DISPLAY FOR NOTEBOOK-FRIENDLY TABLE RENDERING
# ====================================================

from IPython.display import display

# ===================================================================
# IMPORT XGBOOST REGRESSOR FOR DAG-CONSTRAINED RISK FORECAST MODELING
# ===================================================================

from xgboost import XGBRegressor

# ================================================================
# IMPORT SKLEARN ERROR METRICS FOR MAE AND ROOT MEAN SQUARED ERROR
# ================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

# ====================================================================
# SET GLOBAL RANDOM SEED TO MATCH THE CAPSTONE TRACEABILITY CONVENTION
# ====================================================================

RANDOM_SEED = 692

# ===============================================================
# APPLY RANDOM SEED TO NUMPY FOR DETERMINISTIC NUMERICAL SAMPLING
# ===============================================================

np.random.seed(RANDOM_SEED)

# ===================================================================
# APPLY RANDOM SEED TO PYTHON RANDOM FOR DETERMINISTIC PARAMETER HANDLING
# ===================================================================

random.seed(RANDOM_SEED)

# =========================================================================
# FREEZE FORECAST HORIZON AT TWENTY TRADING DAYS TO MATCH NOTEBOOK 03
# =========================================================================

FORECAST_HORIZON_DAYS = 20

# ===========================================================================
# FREEZE WALK-FORWARD STEP AT TWENTY TRADING DAYS TO MATCH NOTEBOOK 03
# ===========================================================================

WALK_FORWARD_STEP_DAYS = 20

# ===========================================================================
# DEFINE ANNUALIZATION FACTOR USING SQUARE ROOT OF TWO HUNDRED FIFTY TWO
# ===========================================================================

ANNUALIZATION_FACTOR = float(np.sqrt(252.0))

# ========================================================================
# CONFIGURE PANDAS DISPLAY ROW LIMIT FOR AUDITABLE NOTEBOOK TABLE PREVIEWS
# ========================================================================

pd.set_option("display.max_rows", 30)

# =====================================================================
# CONFIGURE PANDAS DISPLAY COLUMN LIMIT FOR WIDE FEATURE TABLE PREVIEWS
# =====================================================================

pd.set_option("display.max_columns", 40)

# =====================================================================
# CONFIGURE CONSISTENT FLOAT DISPLAY FORMAT FOR METRIC COMPARISON CELLS
# =====================================================================

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

# =====================================================================
# CAPTURE CURRENT WORKING DIRECTORY AS THE FIRST PROJECT ROOT CANDIDATE
# =====================================================================

CWD = Path.cwd().resolve()

# =============================================================================
# ASSEMBLE FALLBACK ROOT CANDIDATES FOR LOCAL REPOSITORY AND GOOGLE COLAB PATHS
# =============================================================================

PROJECT_ROOT_CANDIDATES = []
PROJECT_ROOT_CANDIDATES.extend([CWD, *list(CWD.parents)])
PROJECT_ROOT_CANDIDATES.extend(
    [
        Path("/content/riskml-capstone"),
        Path("/content/drive/MyDrive/MScFE_692_Capstone/riskml_repo"),
        Path("/content/drive/MyDrive/riskml-capstone"),
    ]
)

# ================================================================================
# DETECT PROJECT ROOT BY FINDING THE FIRST CANDIDATE CONTAINING THE DATA DIRECTORY
# ================================================================================

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "data").exists()), None)

# ==========================================================
# RAISE A CLEAR FILE ERROR WHEN PROJECT ROOT DETECTION FAILS
# ==========================================================

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "PROJECT ROOT DETECTION FAILED: EXPECTED A REPOSITORY ROOT CONTAINING DATA DIRECTORY"
    )

# ===============================================================================
# DEFINE CANONICAL DATA INPUT PATHS USED BY NOTEBOOK 02 AND NOTEBOOK 03 ARTIFACTS
# ===============================================================================

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_PATH = DATA_PROCESSED_DIR / "features.parquet"
TARGET_PATH = DATA_PROCESSED_DIR / "target_fwd_vol.parquet"
RETURNS_PATH = DATA_PROCESSED_DIR / "etf_returns.parquet"

# ===============================================================================
# DEFINE CANONICAL NOTEBOOK 03 ARTIFACT PATHS REQUIRED FOR NOTEBOOK 04 COMPARISON
# ===============================================================================

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
MODELS_DIR = REPORTS_DIR / "models"
NB03_BASELINE_METRICS_PATH = TABLES_DIR / "notebook03_baseline_rmse_mae.csv"
NB03_BASELINE_PREDICTIONS_PATH = TABLES_DIR / "notebook03_baseline_predictions_long.csv"
NB03_BASELINE_TUNING_PATH = TABLES_DIR / "notebook03_xgb_tuning_results.csv"
NB03_BASELINE_IMPORTANCE_PATH = TABLES_DIR / "notebook03_xgb_feature_importance.csv"
NB03_BASELINE_MODEL_PATH = MODELS_DIR / "notebook03_xgb_baseline.json"

# =========================================================================
# DEFINE NOTEBOOK 04 OUTPUT PATHS FOR TABLES FIGURES MODELS AND PREDICTIONS
# =========================================================================

NB04_CAUSAL_METRICS_PATH = TABLES_DIR / "notebook04_causal_rmse_mae.csv"
NB04_CAUSAL_PREDICTIONS_TABLE_PATH = TABLES_DIR / "notebook04_causal_predictions_long.csv"
NB04_CAUSAL_PREDICTIONS_DATA_PATH = DATA_PROCESSED_DIR / "notebook04_causal_predictions_long.csv"
NB04_GATING_SUMMARY_PATH = TABLES_DIR / "notebook04_dag_gating_summary.csv"
NB04_EFFECTIVE_WINDOW_PATH = TABLES_DIR / "notebook04_effective_window_summary.csv"
NB04_SPLIT_SUMMARY_PATH = TABLES_DIR / "notebook04_split_summary.csv"
NB04_COMPARISON_PATH = TABLES_DIR / "causal_vs_baseline_rmse_mae.csv"
NB04_FEATURE_IMPORTANCE_PATH = TABLES_DIR / "notebook04_causal_feature_importance.csv"
NB04_ENTROPY_PATH = TABLES_DIR / "notebook04_entropy_comparison.csv"
NB04_PREFIX_GAIN_PATH = TABLES_DIR / "notebook04_prefix_gain_share.csv"
NB04_TOP_FEATURE_COMPARISON_PATH = TABLES_DIR / "notebook04_top_feature_comparison.csv"
NB04_ASSET_CLASS_PATH = TABLES_DIR / "notebook04_asset_class_comparison.csv"
NB04_ARTIFACT_MANIFEST_PATH = TABLES_DIR / "notebook04_artifact_manifest.csv"

# ===================================================================
# CREATE REPORT OUTPUT DIRECTORIES WITH IDEMPOTENT DIRECTORY CREATION
# ===================================================================

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# =======================================================================
# DEFINE ROOT MEAN SQUARED ERROR HELPER FOR CONSISTENT METRIC COMPUTATION
# =======================================================================

def compute_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # =========================================
    # RETURN ROOT MEAN SQUARED ERROR AS A FLOAT
    # =========================================

    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# =========================================================================
# DEFINE OPTIONAL CSV LOADER THAT RETURNS NONE WHEN THE FILE DOES NOT EXIST
# =========================================================================

def load_optional_csv(path: Path) -> Optional[pd.DataFrame]:
    # ======================================================
    # RETURN DATAFRAME WHEN THE CSV EXISTS OR NONE OTHERWISE
    # ======================================================

    return pd.read_csv(path) if path.exists() else None

# =========================================================================
# DEFINE CSV EXPORT HELPER WITH DIRECTORY SAFETY FOR REPORT TABLE ARTIFACTS
# =========================================================================

def save_dataframe_csv(df: pd.DataFrame, path: Path) -> None:
    # ================================================
    # ENSURE PARENT DIRECTORY EXISTS BEFORE CSV EXPORT
    # ================================================

    path.parent.mkdir(parents=True, exist_ok=True)

    # ===============================================================
    # WRITE DATAFRAME TO CSV WITHOUT INDEX FOR CLEAN REPORT INGESTION
    # ===============================================================

    df.to_csv(path, index=False)

# ============================================================================
# DEFINE PNG EXPORT HELPER WITH REPORT-READY RESOLUTION AND TIGHT BOUNDING BOX
# ============================================================================

def save_figure_png(fig: plt.Figure, path: Path, dpi: int = 200) -> None:
    # ===================================================
    # ENSURE PARENT DIRECTORY EXISTS BEFORE FIGURE EXPORT
    # ===================================================

    path.parent.mkdir(parents=True, exist_ok=True)

    # =============================================================
    # SAVE FIGURE TO PNG WITH TIGHT BOUNDING BOX FOR WORD INSERTION
    # =============================================================

    fig.savefig(path, dpi=dpi, bbox_inches="tight")

# ======================================================================
# PRINT ENVIRONMENT AND PATH AUDIT OUTPUT FOR THE FIRST NOTEBOOK 04 CELL
# ======================================================================

print("PYTHON_VERSION:", sys.version.split()[0])
print("CURRENT_WORKING_DIRECTORY:", str(CWD))
print("PROJECT_ROOT:", str(PROJECT_ROOT))
print("FEATURES_PATH_EXISTS:", FEATURES_PATH.exists())
print("TARGET_PATH_EXISTS:", TARGET_PATH.exists())
print("RETURNS_PATH_EXISTS:", RETURNS_PATH.exists())
print("NB03_BASELINE_METRICS_PATH_EXISTS:", NB03_BASELINE_METRICS_PATH.exists())
print("NB03_BASELINE_PREDICTIONS_PATH_EXISTS:", NB03_BASELINE_PREDICTIONS_PATH.exists())
print("NB03_BASELINE_IMPORTANCE_PATH_EXISTS:", NB03_BASELINE_IMPORTANCE_PATH.exists())

PYTHON_VERSION: 3.12.12
CURRENT_WORKING_DIRECTORY: /content/riskml-capstone
PROJECT_ROOT: /content/riskml-capstone
FEATURES_PATH_EXISTS: True
TARGET_PATH_EXISTS: True
RETURNS_PATH_EXISTS: True
NB03_BASELINE_METRICS_PATH_EXISTS: True
NB03_BASELINE_PREDICTIONS_PATH_EXISTS: True
NB03_BASELINE_IMPORTANCE_PATH_EXISTS: True


### 📌 Observations and Insights for CODE_BLOCK_A

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — All three upstream Parquet files exist. All three Notebook 03 baseline artifacts (metrics, predictions, importance) are available for downstream comparison.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Imported all required libraries (NumPy, Pandas, XGBoost, scikit-learn, Matplotlib) and suppressed non-critical warnings for clean notebook output.
- Set the global random seed to 692 for both NumPy and Python `random`, matching the capstone traceability convention established in Notebooks 01–03.
- Froze the forecast horizon (20 trading days) and walk-forward step size (20 trading days) as constants to match the Notebook 03 evaluation protocol exactly.
- Detected `PROJECT_ROOT` at `/content/riskml-capstone` by scanning candidate paths for a directory containing `data/`.
- Defined all canonical input paths (features, targets, returns) and all Notebook 03 artifact paths required for the baseline comparison in CODE_BLOCK_G.
- Defined all 13 Notebook 04 output paths for tables, figures, models, and predictions — establishing the full artifact contract before any computation begins.
- Created `reports/figures/`, `reports/tables/`, `reports/models/`, and `data/processed/` directories idempotently.
- Defined three reusable helper functions: `compute_rmse()`, `load_optional_csv()`, `save_dataframe_csv()`, and `save_figure_png()`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `PYTHON_VERSION`: 3.12.12 (Colab runtime)
- `PROJECT_ROOT`: `/content/riskml-capstone`
- `FEATURES_PATH_EXISTS`: True
- `TARGET_PATH_EXISTS`: True
- `RETURNS_PATH_EXISTS`: True
- `NB03_BASELINE_METRICS_PATH_EXISTS`: True
- `NB03_BASELINE_PREDICTIONS_PATH_EXISTS`: True
- `NB03_BASELINE_IMPORTANCE_PATH_EXISTS`: True
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> All six critical file-existence checks returned True — the Notebook 02 data pipeline and Notebook 03 baseline artifacts are fully available for Notebook 04 consumption.
- <span style="color: darkgreen;"><strong>✓</strong></span> The `PROJECT_ROOT` detection correctly resolved to `/content/riskml-capstone` via the Colab clone path, confirming that the `git clone` setup cell executed successfully before CODE_BLOCK_A.
- <span style="color: darkgreen;"><strong>✓</strong></span> Random seed 692 is applied to both NumPy and Python `random` before any stochastic operation, satisfying the Fresh-Kernel Reliability Rule (Handoff Section 11.8).
</div>

**Next step:** CODE_BLOCK_B loads the three Parquet files and the four optional Notebook 03 CSV artifacts, validates index alignment, drops the 100%-NaN sentiment placeholder, and confirms the 14-ticker universe.


***

In [14]:
# ============
# CODE_BLOCK_B
# ============

# ==================================================================
# LOAD FEATURES TARGETS AND RETURNS GENERATED BY THE PRIOR NOTEBOOKS
# ==================================================================

FEATURES_DF = pd.read_parquet(FEATURES_PATH, engine="pyarrow")
TARGET_DF = pd.read_parquet(TARGET_PATH, engine="pyarrow")
RETURNS_DF = pd.read_parquet(RETURNS_PATH, engine="pyarrow")

# ==================================================================
# LOAD OPTIONAL NOTEBOOK 03 ARTIFACTS FOR DIRECT BASELINE COMPARISON
# ==================================================================

NB03_BASELINE_METRICS_DF = load_optional_csv(NB03_BASELINE_METRICS_PATH)
NB03_BASELINE_PREDICTIONS_DF = load_optional_csv(NB03_BASELINE_PREDICTIONS_PATH)
NB03_BASELINE_TUNING_DF = load_optional_csv(NB03_BASELINE_TUNING_PATH)
NB03_BASELINE_IMPORTANCE_DF = load_optional_csv(NB03_BASELINE_IMPORTANCE_PATH)

# ========================================================================
# COERCE ALL PRIMARY INDICES TO DATETIME FOR TIME-ORDERED ALIGNMENT CHECKS
# ========================================================================

FEATURES_DF.index = pd.to_datetime(FEATURES_DF.index)
TARGET_DF.index = pd.to_datetime(TARGET_DF.index)
RETURNS_DF.index = pd.to_datetime(RETURNS_DF.index)

# ==============================================================
# SORT ALL PRIMARY DATAFRAMES TO ENFORCE MONOTONIC TIME ORDERING
# ==============================================================

FEATURES_DF = FEATURES_DF.sort_index()
TARGET_DF = TARGET_DF.sort_index()
RETURNS_DF = RETURNS_DF.sort_index()

# =============================================================================
# ASSERT INDEX ALIGNMENT ACROSS FEATURES TARGETS AND RETURNS TO PREVENT LEAKAGE
# =============================================================================

if not FEATURES_DF.index.equals(TARGET_DF.index):
    raise ValueError("INDEX MISALIGNMENT: FEATURES_DF INDEX DOES NOT MATCH TARGET_DF INDEX")

if not FEATURES_DF.index.equals(RETURNS_DF.index):
    raise ValueError("INDEX MISALIGNMENT: FEATURES_DF INDEX DOES NOT MATCH RETURNS_DF INDEX")

# =======================================================================
# IDENTIFY SENTIMENT PLACEHOLDER COLUMNS FOR EXPLICIT AUDIT AND EXCLUSION
# =======================================================================

SENTIMENT_COLS = [col for col in FEATURES_DF.columns if col.startswith("SENT__")]
SENTIMENT_NAN_FRACTION = float(FEATURES_DF[SENTIMENT_COLS].isna().mean().mean()) if len(SENTIMENT_COLS) > 0 else 0.0

# ===========================================================================
# DROP SENTIMENT PLACEHOLDER COLUMNS TO REUSE NOTEBOOK 03 MODELING CONVENTION
# ===========================================================================

FEATURES_NO_SENT_DF = FEATURES_DF.drop(columns=SENTIMENT_COLS, errors="ignore")

# ============================================================
# CHECK FOR FORBIDDEN TARGET COLUMNS INSIDE THE FEATURE MATRIX
# ============================================================

LEAKAGE_COLUMNS = [col for col in FEATURES_NO_SENT_DF.columns if col.startswith("TARGET__")]

if len(LEAKAGE_COLUMNS) > 0:
    raise ValueError(f"LEAKAGE DETECTED: TARGET-PREFIXED COLUMNS FOUND IN FEATURES: {LEAKAGE_COLUMNS[:10]}")

# =======================================================================
# CAPTURE TICKER LIST FROM RETURNS COLUMNS FOR CONSISTENT ITERATION ORDER
# =======================================================================

TICKER_LIST = list(RETURNS_DF.columns)

# =====================================================================
# SUMMARIZE FEATURE PREFIX DISTRIBUTION BEFORE EFFECTIVE WINDOW MASKING
# =====================================================================

PREFIX_COUNTS_BEFORE_MASK_DF = (
    FEATURES_NO_SENT_DF.columns.to_series()
    .map(lambda col: col.split("__")[0] if "__" in col else "UNSCOPED")
    .value_counts()
    .rename_axis("dag_prefix")
    .reset_index(name="feature_count")
    .sort_values(["dag_prefix"])
    .reset_index(drop=True)
)

# ================================================================
# PRINT INPUT SHAPES DATE RANGE AND OPTIONAL ARTIFACT AVAILABILITY
# ================================================================

print("FEATURES_DF_SHAPE:", FEATURES_DF.shape)
print("FEATURES_NO_SENT_DF_SHAPE:", FEATURES_NO_SENT_DF.shape)
print("TARGET_DF_SHAPE:", TARGET_DF.shape)
print("RETURNS_DF_SHAPE:", RETURNS_DF.shape)
print("DATE_RANGE_START:", str(FEATURES_DF.index.min().date()))
print("DATE_RANGE_END:", str(FEATURES_DF.index.max().date()))
print("SENTIMENT_COLS:", SENTIMENT_COLS)
print("SENTIMENT_NAN_FRACTION:", f"{SENTIMENT_NAN_FRACTION:0.6f}")
print("TICKER_LIST:", TICKER_LIST)
print("NB03_BASELINE_METRICS_AVAILABLE:", NB03_BASELINE_METRICS_DF is not None)
print("NB03_BASELINE_PREDICTIONS_AVAILABLE:", NB03_BASELINE_PREDICTIONS_DF is not None)
print("NB03_BASELINE_TUNING_AVAILABLE:", NB03_BASELINE_TUNING_DF is not None)
print("NB03_BASELINE_IMPORTANCE_AVAILABLE:", NB03_BASELINE_IMPORTANCE_DF is not None)

# ============================================================================
# DISPLAY SMALL TABLE PREVIEWS FOR HUMAN AUDIT AND NEXT MARKDOWN INSIGHT CELLS
# ============================================================================

display(FEATURES_NO_SENT_DF.iloc[:3, :10])
display(TARGET_DF.iloc[:3, :5])
display(RETURNS_DF.iloc[:3, :5])
display(PREFIX_COUNTS_BEFORE_MASK_DF)

FEATURES_DF_SHAPE: (2798, 152)
FEATURES_NO_SENT_DF_SHAPE: (2798, 151)
TARGET_DF_SHAPE: (2798, 14)
RETURNS_DF_SHAPE: (2798, 14)
DATE_RANGE_START: 2015-01-05
DATE_RANGE_END: 2026-02-19
SENTIMENT_COLS: ['SENT__sentiment_market']
SENTIMENT_NAN_FRACTION: 1.000000
TICKER_LIST: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'XLK', 'XLF', 'XLE', 'XLV', 'TLT', 'LQD', 'HYG', 'GLD', 'DBC']
NB03_BASELINE_METRICS_AVAILABLE: True
NB03_BASELINE_PREDICTIONS_AVAILABLE: True
NB03_BASELINE_TUNING_AVAILABLE: True
NB03_BASELINE_IMPORTANCE_AVAILABLE: True


,MOM__SPY__cum_ret__5d,MOM__QQQ__cum_ret__5d,MOM__IWM__cum_ret__5d,MOM__EFA__cum_ret__5d,MOM__EEM__cum_ret__5d,MOM__XLK__cum_ret__5d,MOM__XLF__cum_ret__5d,MOM__XLE__cum_ret__5d,MOM__XLV__cum_ret__5d,MOM__TLT__cum_ret__5d
Date,,,,,,,,,,
2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,TARGET__SPY__fwd_rvol__20d,TARGET__QQQ__fwd_rvol__20d,TARGET__IWM__fwd_rvol__20d,TARGET__EFA__fwd_rvol__20d,TARGET__EEM__fwd_rvol__20d
Date,,,,,
2015-01-05,0.170624,0.187530,0.209809,0.157650,0.209774
2015-01-06,0.167277,0.180012,0.199511,0.154217,0.210131
2015-01-07,0.165369,0.177073,0.199956,0.156246,0.199064


Ticker,SPY,QQQ,IWM,EFA,EEM
Date,,,,,
2015-01-05,-0.018225,-0.014777,-0.013460,-0.023888,-0.017958
2015-01-06,-0.009464,-0.013499,-0.017451,-0.011391,-0.004210
2015-01-07,0.012384,0.012808,0.012239,0.011054,0.021394


,dag_prefix,feature_count
0,MACRO,3
1,ML,3
2,MOM,56
3,REGIME,1
4,VAL,18
5,VOL,70


### 📌 Observations and Insights for CODE_BLOCK_B

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Index alignment confirmed across all three primary DataFrames. No TARGET__-prefixed leakage columns detected in the feature matrix. All four Notebook 03 artifacts loaded successfully.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Loaded `features.parquet` (2,798 × 152), `target_fwd_vol.parquet` (2,798 × 14), and `etf_returns.parquet` (2,798 × 14) from the `data/processed/` directory.
- Loaded four optional Notebook 03 CSV artifacts: baseline metrics, baseline predictions, tuning results, and feature importance — all returned non-None.
- Coerced all primary DataFrame indices to datetime and enforced monotonic time ordering via `sort_index()`.
- Validated strict index alignment: `FEATURES_DF.index.equals(TARGET_DF.index)` and `FEATURES_DF.index.equals(RETURNS_DF.index)` — both passed without raising ValueError.
- Identified and dropped the `SENT__sentiment_market` column (100% NaN placeholder), reducing the feature matrix from 152 to 151 columns.
- Confirmed that zero TARGET__-prefixed columns exist inside the feature matrix — the physical leakage boundary between features and targets holds.
- Captured the 14-ticker universe from `RETURNS_DF.columns` for consistent iteration ordering across all downstream loops.
- Computed prefix-count distribution before effective-window masking: MOM (56), VOL (70), VAL (18), MACRO (3), ML (3), REGIME (1) = 151 total.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `FEATURES_DF_SHAPE`: (2798, 152) → `FEATURES_NO_SENT_DF_SHAPE`: (2798, 151) after sentiment exclusion
- `DATE_RANGE`: 2015-01-05 through 2026-02-19 (2,798 trading days)
- `SENTIMENT_NAN_FRACTION`: 1.000000 (confirmed 100% NaN)
- `TICKER_LIST`: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'XLK', 'XLF', 'XLE', 'XLV', 'TLT', 'LQD', 'HYG', 'GLD', 'DBC']
- All four NB03 artifacts: Available = True
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The 152 → 151 column reduction confirms that exactly one sentiment column was dropped, matching the Notebook 03 convention and the handoff specification (Section 5.2.2: "SENT-prefixed (1 column, 100% NaN) is also excluded but already dropped from modeling").
- <span style="color: darkgreen;"><strong>✓</strong></span> The prefix distribution (MOM=56, VOL=70, VAL=18, MACRO=3, ML=3, REGIME=1) sums to 151 and matches the feature contract documented in `docs/feature_contract.csv`.
- <span style="color: darkgreen;"><strong>✓</strong></span> All four Notebook 03 artifacts loaded successfully — CODE_BLOCK_G will merge causal metrics against these baselines for the primary H1 comparison.
- <span style="color: darkorange;"><strong>⚠</strong></span> The first three rows of the feature matrix display NaN values (visible in the MOM__*__cum_ret__5d columns). These warm-up NaN rows will be removed by the effective modeling mask in CODE_BLOCK_C — no action required at this stage.
</div>

**Next step:** CODE_BLOCK_C applies the effective modeling mask (simultaneous non-NaN across all features and all targets) and constructs the TARGET_COL_MAP governance dictionary for safe target access.


***

In [15]:
# ============
# CODE_BLOCK_C
# ============

# =========================================================================
# COMPUTE ROW-WISE COMPLETENESS MASK FOR FEATURES AFTER SENTIMENT EXCLUSION
# =========================================================================

FEATURES_COMPLETE_MASK = FEATURES_NO_SENT_DF.notna().all(axis=1)

# =============================================================
# COMPUTE ROW-WISE COMPLETENESS MASK FOR THE FULL TARGET MATRIX
# =============================================================

TARGET_COMPLETE_MASK = TARGET_DF.notna().all(axis=1)

# ====================================================================
# INTERSECT FEATURE AND TARGET COMPLETENESS MASKS TO MATCH NOTEBOOK 03
# ====================================================================

EFFECTIVE_MODELING_MASK = FEATURES_COMPLETE_MASK & TARGET_COMPLETE_MASK

# =================================================================
# APPLY THE EFFECTIVE MODELING MASK TO FEATURES TARGETS AND RETURNS
# =================================================================

EFF_FEATURES_DF = FEATURES_NO_SENT_DF.loc[EFFECTIVE_MODELING_MASK].copy()
EFF_TARGET_DF = TARGET_DF.loc[EFFECTIVE_MODELING_MASK].copy()
EFF_RETURNS_DF = RETURNS_DF.loc[EFFECTIVE_MODELING_MASK].copy()

# =========================================================================
# EXTRACT EFFECTIVE WINDOW DATES AND COUNTS FOR THE NOTEBOOK 04 AUDIT TABLE
# =========================================================================

EFFECTIVE_START_DATE = EFF_FEATURES_DF.index.min()
EFFECTIVE_END_DATE = EFF_FEATURES_DF.index.max()
EFFECTIVE_ROW_COUNT = int(EFF_FEATURES_DF.shape[0])
EFFECTIVE_FEATURE_COUNT = int(EFF_FEATURES_DF.shape[1])
EFFECTIVE_TARGET_COUNT = int(EFF_TARGET_DF.shape[1])
EFFECTIVE_FEATURES_MEAN_NON_NULL = float(EFF_FEATURES_DF.notna().mean().mean())
EFFECTIVE_TARGETS_MEAN_NON_NULL = float(EFF_TARGET_DF.notna().mean().mean())

# =====================================================================
# BUILD TICKER-TO-TARGET-COLUMN MAP FOR SAFE DAG-PREFIXED TARGET ACCESS
# =====================================================================

TARGET_COL_MAP: Dict[str, str] = {}

for ticker in TICKER_LIST:
    # ======================================
    # MATCH DAG-PREFIXED TARGET COLUMN FIRST
    # ======================================

    matched_cols = [col for col in EFF_TARGET_DF.columns if f"TARGET__{ticker}__" in col]

    # ==================================================================
    # ASSIGN THE FIRST MATCHED DAG-PREFIXED TARGET COLUMN WHEN AVAILABLE
    # ==================================================================

    if len(matched_cols) > 0:
        TARGET_COL_MAP[ticker] = matched_cols[0]

    # ==========================================================================
    # FALL BACK TO SIMPLE TICKER COLUMN NAME WHEN THE TARGET IS NOT DAG-PREFIXED
    # ==========================================================================

    elif ticker in EFF_TARGET_DF.columns:
        TARGET_COL_MAP[ticker] = ticker

    # ====================================================
    # RAISE AN ERROR WHEN NO TARGET COLUMN CAN BE RESOLVED
    # ====================================================

    else:
        raise KeyError(f"TARGET COLUMN NOT FOUND FOR TICKER={ticker}")

# ====================================================================
# SUMMARIZE FEATURE PREFIX DISTRIBUTION AFTER EFFECTIVE WINDOW MASKING
# ====================================================================

PREFIX_COUNTS_EFFECTIVE_DF = (
    EFF_FEATURES_DF.columns.to_series()
    .map(lambda col: col.split("__")[0] if "__" in col else "UNSCOPED")
    .value_counts()
    .rename_axis("dag_prefix")
    .reset_index(name="feature_count")
    .sort_values(["dag_prefix"])
    .reset_index(drop=True)
)

# =========================================================================
# BUILD EFFECTIVE WINDOW SUMMARY TABLE FOR NOTEBOOK 04 ARTIFACT PERSISTENCE
# =========================================================================

EFFECTIVE_WINDOW_SUMMARY_DF = pd.DataFrame(
    [
        {
            "effective_start_date": str(EFFECTIVE_START_DATE.date()),
            "effective_end_date": str(EFFECTIVE_END_DATE.date()),
            "effective_row_count": EFFECTIVE_ROW_COUNT,
            "effective_feature_count_ex_sentiment": EFFECTIVE_FEATURE_COUNT,
            "effective_target_count": EFFECTIVE_TARGET_COUNT,
            "effective_features_mean_non_null_fraction": EFFECTIVE_FEATURES_MEAN_NON_NULL,
            "effective_targets_mean_non_null_fraction": EFFECTIVE_TARGETS_MEAN_NON_NULL,
        }
    ]
)

# ==================================================================
# SAVE EFFECTIVE WINDOW SUMMARY FOR DOWNSTREAM NOTEBOOK TRACEABILITY
# ==================================================================

save_dataframe_csv(EFFECTIVE_WINDOW_SUMMARY_DF, NB04_EFFECTIVE_WINDOW_PATH)

# ================================================================
# PRINT EFFECTIVE WINDOW AUDIT OUTPUT AND TARGET COLUMN MAP SAMPLE
# ================================================================

print("EFFECTIVE_START_DATE:", str(EFFECTIVE_START_DATE.date()))
print("EFFECTIVE_END_DATE:", str(EFFECTIVE_END_DATE.date()))
print("EFFECTIVE_ROW_COUNT:", EFFECTIVE_ROW_COUNT)
print("EFFECTIVE_FEATURE_COUNT_EX_SENTIMENT:", EFFECTIVE_FEATURE_COUNT)
print("EFFECTIVE_TARGET_COUNT:", EFFECTIVE_TARGET_COUNT)
print("NB04_EFFECTIVE_WINDOW_PATH:", str(NB04_EFFECTIVE_WINDOW_PATH))
print("TARGET_COL_MAP:", json.dumps(TARGET_COL_MAP, indent=2))

# ==============================================================================
# DISPLAY EFFECTIVE WINDOW SUMMARY AND PREFIX COUNT TABLES FOR NOTEBOOK INSIGHTS
# ==============================================================================

display(EFFECTIVE_WINDOW_SUMMARY_DF)
display(PREFIX_COUNTS_EFFECTIVE_DF)

EFFECTIVE_START_DATE: 2016-01-04
EFFECTIVE_END_DATE: 2025-12-31
EFFECTIVE_ROW_COUNT: 2496
EFFECTIVE_FEATURE_COUNT_EX_SENTIMENT: 151
EFFECTIVE_TARGET_COUNT: 14
NB04_EFFECTIVE_WINDOW_PATH: /content/riskml-capstone/reports/tables/notebook04_effective_window_summary.csv
TARGET_COL_MAP: {
  "SPY": "TARGET__SPY__fwd_rvol__20d",
  "QQQ": "TARGET__QQQ__fwd_rvol__20d",
  "IWM": "TARGET__IWM__fwd_rvol__20d",
  "EFA": "TARGET__EFA__fwd_rvol__20d",
  "EEM": "TARGET__EEM__fwd_rvol__20d",
  "XLK": "TARGET__XLK__fwd_rvol__20d",
  "XLF": "TARGET__XLF__fwd_rvol__20d",
  "XLE": "TARGET__XLE__fwd_rvol__20d",
  "XLV": "TARGET__XLV__fwd_rvol__20d",
  "TLT": "TARGET__TLT__fwd_rvol__20d",
  "LQD": "TARGET__LQD__fwd_rvol__20d",
  "HYG": "TARGET__HYG__fwd_rvol__20d",
  "GLD": "TARGET__GLD__fwd_rvol__20d",
  "DBC": "TARGET__DBC__fwd_rvol__20d"
}


,effective_start_date,effective_end_date,effective_row_count,effective_feature_count_ex_sentiment,effective_target_count,effective_features_mean_non_null_fraction,effective_targets_mean_non_null_fraction
0,2016-01-04,2025-12-31,2496,151,14,1.000000,1.000000


,dag_prefix,feature_count
0,MACRO,3
1,ML,3
2,MOM,56
3,REGIME,1
4,VAL,18
5,VOL,70


### 📌 Observations and Insights for CODE_BLOCK_C

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Effective modeling window covers 2,496 usable trading days with 100% non-null fractions across both features and targets. TARGET_COL_MAP resolved all 14 tickers to DAG-prefixed column names without error.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Computed row-wise completeness masks for features (151 columns, post-sentiment exclusion) and targets (14 columns) independently, then intersected the masks to produce `EFFECTIVE_MODELING_MASK`.
- Applied the mask to produce `EFF_FEATURES_DF` (2,496 × 151), `EFF_TARGET_DF` (2,496 × 14), and `EFF_RETURNS_DF` (2,496 × 14).
- Built `TARGET_COL_MAP` — a governance dictionary that routes every ticker symbol to the corresponding `TARGET__<TICKER>__fwd_rvol__20d` column name — enforcing the Target-Column Governance Rule (Handoff Section 11.9).
- Computed prefix-count distribution after masking and confirmed that column counts remain unchanged (masking removes rows, not columns).
- Saved `notebook04_effective_window_summary.csv` to `reports/tables/` for artifact traceability.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `EFFECTIVE_START_DATE`: 2016-01-04
- `EFFECTIVE_END_DATE`: 2025-12-31
- `EFFECTIVE_ROW_COUNT`: 2,496
- `EFFECTIVE_FEATURE_COUNT_EX_SENTIMENT`: 151
- `EFFECTIVE_TARGET_COUNT`: 14
- `effective_features_mean_non_null_fraction`: 1.000000
- `effective_targets_mean_non_null_fraction`: 1.000000
- Saved artifact: `notebook04_effective_window_summary.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The effective window (2016-01-04 through 2025-12-31, 2,496 rows) matches the Notebook 03 effective window specification. The leading boundary is determined by the 252-day PCA warm-up; the trailing boundary is determined by the 20-day forward target horizon.
- <span style="color: darkgreen;"><strong>✓</strong></span> Both non-null fractions equal 1.000000, confirming that the effective modeling mask successfully excludes all warm-up and trailing NaN rows — no partial-NaN rows survive into the modeling window.
- <span style="color: darkgreen;"><strong>✓</strong></span> All 14 tickers resolved through `TARGET_COL_MAP` to DAG-prefixed column names (e.g., `SPY` → `TARGET__SPY__fwd_rvol__20d`), preventing the KeyError/silent-NaN bug class documented in the handoff (Section 11.9).
- <span style="color: darkgreen;"><strong>✓</strong></span> The 302 excluded rows (2,798 − 2,496 = 302) account for approximately 251 leading warm-up rows (PCA binding constraint) plus approximately 20 trailing target-horizon rows plus scattered FF/FRED missing dates — consistent with the structural missingness sources documented in Handoff Section 6.1.
</div>

**Next step:** CODE_BLOCK_D defines the manual DAG edge list, applies prefix-based feature gating for the Risk forecast stage, and validates that no forbidden-prefix features leaked through the constraint layer.


***

In [16]:
# ============
# CODE_BLOCK_D
# ============

# =====================================================================
# DEFINE THE MANUAL DAG EDGE LIST AS A DIRECTIONAL CONSTRAINT REFERENCE
# =====================================================================

CAUSAL_EDGES = {
    "SENTIMENT": ["MOMENTUM"],
    "MOMENTUM": ["RETURNS"],
    "VALUE": ["RETURNS"],
    "VOLATILITY": ["RISK"],
    "RISK": ["ALLOCATION"],
}

# ================================================================
# DEFINE PREFIX MAP USED TO TRANSLATE FEATURE NAMES INTO DAG NODES
# ================================================================

NODE_PREFIX_MAP = {
    "SENTIMENT": "SENT__",
    "MOMENTUM": "MOM__",
    "VALUE": "VAL__",
    "VOLATILITY": "VOL__",
    "MACRO": "MACRO__",
    "ML": "ML__",
    "REGIME": "REGIME__",
    "TARGET": "TARGET__",
}

# ====================================================================
# FREEZE ALLOWED AND FORBIDDEN PREFIX SETS FOR THE RISK FORECAST STAGE
# ====================================================================

ALLOWED_PREFIXES_RISK = ["VOL__", "MACRO__", "REGIME__"]
FORBIDDEN_PREFIXES_RISK = ["MOM__", "VAL__", "ML__", "SENT__"]

# ========================================================================
# DEFINE HELPER TO EXTRACT THE DAG PREFIX TOKEN FROM A FEATURE COLUMN NAME
# ========================================================================

def extract_feature_prefix(column_name: str) -> str:
    # ==============================================================================
    # RETURN THE FIRST DOUBLE-UNDERSCORE TOKEN OR UNSCOPED WHEN THE TOKEN IS MISSING
    # ==============================================================================

    return column_name.split("__")[0] if "__" in column_name else "UNSCOPED"

# ========================================================
# DEFINE HELPER TO GATE FEATURES BY AN ALLOWED PREFIX LIST
# ========================================================

def gate_features_by_prefix(columns: List[str], allowed_prefixes: List[str]) -> Tuple[List[str], List[str]]:
    # ========================================================
    # COLLECT ALLOWED FEATURE COLUMNS IN ORIGINAL COLUMN ORDER
    # ========================================================

    allowed_cols = [col for col in columns if any(col.startswith(prefix) for prefix in allowed_prefixes)]

    # =========================================================
    # COLLECT EXCLUDED FEATURE COLUMNS IN ORIGINAL COLUMN ORDER
    # =========================================================

    excluded_cols = [col for col in columns if col not in allowed_cols]

    # =============================================
    # RETURN BOTH ALLOWED AND EXCLUDED COLUMN LISTS
    # =============================================

    return allowed_cols, excluded_cols

# =====================================================================
# APPLY PREFIX GATING TO THE EFFECTIVE FEATURE MATRIX FOR THE RISK NODE
# =====================================================================

CAUSAL_FEATURE_COLS, EXCLUDED_FEATURE_COLS = gate_features_by_prefix(list(EFF_FEATURES_DF.columns), ALLOWED_PREFIXES_RISK)

# ====================================================================
# VALIDATE THAT NO FORBIDDEN PREFIX SURVIVED THE RISK NODE GATING STEP
# ====================================================================

LEAKED_FORBIDDEN_COLS = [
    col
    for col in CAUSAL_FEATURE_COLS
    if any(col.startswith(prefix) for prefix in FORBIDDEN_PREFIXES_RISK)
]

if len(LEAKED_FORBIDDEN_COLS) > 0:
    raise ValueError(f"DAG GATING FAILURE: FORBIDDEN FEATURES LEAKED INTO RISK STAGE: {LEAKED_FORBIDDEN_COLS[:10]}")

# ======================================================================
# BUILD GATING SUMMARY TABLE WITH PREFIX COUNTS AND ALLOW EXCLUDE STATUS
# ======================================================================

GATING_SUMMARY_DF = (
    pd.DataFrame({"feature": list(EFF_FEATURES_DF.columns)})
    .assign(
        dag_prefix=lambda df: df["feature"].map(extract_feature_prefix),
        allowed_for_risk=lambda df: df["feature"].isin(CAUSAL_FEATURE_COLS),
    )
)

GATING_PREFIX_SUMMARY_DF = (
    GATING_SUMMARY_DF.groupby(["dag_prefix", "allowed_for_risk"], dropna=False)["feature"]
    .count()
    .reset_index(name="feature_count")
    .sort_values(["allowed_for_risk", "dag_prefix"], ascending=[False, True])
    .reset_index(drop=True)
)

# ==================================================================
# SAVE THE FEATURE-LEVEL GATING SUMMARY FOR NOTEBOOK 04 TRACEABILITY
# ==================================================================

save_dataframe_csv(GATING_SUMMARY_DF, NB04_GATING_SUMMARY_PATH)

# ====================================================================
# CREATE THE CONSTRAINED FEATURE MATRIX USED BY ALL CAUSAL MODEL CELLS
# ====================================================================

CAUSAL_X_DF = EFF_FEATURES_DF.loc[:, CAUSAL_FEATURE_COLS].copy()

# ===================================================================
# PRINT GATING COUNTS AND EXPECTED PREFIX COUNTS FOR QUICK VALIDATION
# ===================================================================

print("CAUSAL_FEATURE_COUNT:", len(CAUSAL_FEATURE_COLS))
print("EXCLUDED_FEATURE_COUNT:", len(EXCLUDED_FEATURE_COLS))
print("ALLOWED_PREFIXES_RISK:", ALLOWED_PREFIXES_RISK)
print("FORBIDDEN_PREFIXES_RISK:", FORBIDDEN_PREFIXES_RISK)
print("LEAKED_FORBIDDEN_COLS:", LEAKED_FORBIDDEN_COLS)
print("NB04_GATING_SUMMARY_PATH:", str(NB04_GATING_SUMMARY_PATH))

# ===============================================================
# DISPLAY PREFIX SUMMARY AND EXAMPLE ALLOWED OR EXCLUDED FEATURES
# ===============================================================

display(GATING_PREFIX_SUMMARY_DF)
display(pd.DataFrame({"allowed_feature_example": CAUSAL_FEATURE_COLS[:20]}))
display(pd.DataFrame({"excluded_feature_example": EXCLUDED_FEATURE_COLS[:20]}))

CAUSAL_FEATURE_COUNT: 74
EXCLUDED_FEATURE_COUNT: 77
ALLOWED_PREFIXES_RISK: ['VOL__', 'MACRO__', 'REGIME__']
FORBIDDEN_PREFIXES_RISK: ['MOM__', 'VAL__', 'ML__', 'SENT__']
LEAKED_FORBIDDEN_COLS: []
NB04_GATING_SUMMARY_PATH: /content/riskml-capstone/reports/tables/notebook04_dag_gating_summary.csv


,dag_prefix,allowed_for_risk,feature_count
0,MACRO,True,3
1,REGIME,True,1
2,VOL,True,70
3,ML,False,3
4,MOM,False,56
5,VAL,False,18


,allowed_feature_example
0,VOL__SPY__rvol__10d
1,VOL__QQQ__rvol__10d
2,VOL__IWM__rvol__10d
3,VOL__EFA__rvol__10d
4,VOL__EEM__rvol__10d
5,VOL__XLK__rvol__10d
6,VOL__XLF__rvol__10d
7,VOL__XLE__rvol__10d
8,VOL__XLV__rvol__10d
9,VOL__TLT__rvol__10d


,excluded_feature_example
0,MOM__SPY__cum_ret__5d
1,MOM__QQQ__cum_ret__5d
2,MOM__IWM__cum_ret__5d
3,MOM__EFA__cum_ret__5d
4,MOM__EEM__cum_ret__5d
5,MOM__XLK__cum_ret__5d
6,MOM__XLF__cum_ret__5d
7,MOM__XLE__cum_ret__5d
8,MOM__XLV__cum_ret__5d
9,MOM__TLT__cum_ret__5d


### 📌 Observations and Insights for CODE_BLOCK_D

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — DAG gating produced exactly 74 allowed features and 77 excluded features. Zero forbidden-prefix columns leaked into the constrained feature set. Gating summary saved to artifact.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Defined the manual DAG edge list encoding five directional constraints: Sentiment → Momentum, Momentum → Returns, Value → Returns, Volatility → Risk, Risk → Allocation.
- Froze the allowed-parent set for the Risk forecast stage: `VOL__` (70 features), `MACRO__` (3 features), `REGIME__` (1 feature) = 74 total allowed features.
- Froze the forbidden-prefix set: `MOM__`, `VAL__`, `ML__`, `SENT__` — all excluded from the Risk node's input.
- Applied the `gate_features_by_prefix()` function to split the 151-column effective feature matrix into 74 allowed and 77 excluded columns.
- Validated that zero columns from forbidden prefixes survived the gating step (`LEAKED_FORBIDDEN_COLS` = empty list).
- Built and saved `notebook04_dag_gating_summary.csv` — a feature-level table documenting each column's DAG prefix and allowed/excluded status.
- Constructed `CAUSAL_X_DF` (2,496 × 74), the constrained feature matrix used by all downstream causal model cells.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `CAUSAL_FEATURE_COUNT`: 74 (VOL=70, MACRO=3, REGIME=1)
- `EXCLUDED_FEATURE_COUNT`: 77 (MOM=56, VAL=18, ML=3)
- `LEAKED_FORBIDDEN_COLS`: [] (empty — gating validated)
- Saved artifact: `notebook04_dag_gating_summary.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The 74 allowed + 77 excluded = 151 total matches the post-sentiment feature count from CODE_BLOCK_B, confirming that no features were dropped or duplicated during the gating process.
- <span style="color: darkgreen;"><strong>✓</strong></span> The feature counts per prefix match the handoff specification exactly (Section 5.2.2): VOL=70, MACRO=3, REGIME=1 allowed; MOM=56, VAL=18, ML=3 excluded.
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The 51% feature reduction (74 out of 151) enforces the Volatility → Risk edge constraint. The constrained model cannot exploit cross-node shortcuts — including the MOM__XLF__cum_ret__10d feature that appeared at rank 2 in the Notebook 03 unconstrained SPY model. Whether the constrained model maintains or improves RMSE without momentum cross-contamination is the central empirical question tested in CODE_BLOCK_F.
- <span style="color: darkgreen;"><strong>✓</strong></span> The allowed-feature example list confirms that only `VOL__` prefixed columns (e.g., `VOL__SPY__rvol__10d`) appear in the first 20 entries, and the excluded-feature example list confirms that only `MOM__` prefixed columns appear — visual proof of correct prefix separation.
</div>

**Next step:** CODE_BLOCK_E defines the 60/20/20 time-ordered train–validation–test split on the constrained feature matrix and builds the expanding-window block indices for walk-forward evaluation.


***

In [17]:
# ============
# CODE_BLOCK_E
# ============

# ===================================================================
# COMPUTE EFFECTIVE SAMPLE LENGTH FOR THE TRAIN VALIDATION TEST SPLIT
# ===================================================================

N_EFFECTIVE = int(len(CAUSAL_X_DF))

# ===============================================================
# RECREATE THE NOTEBOOK 03 SIXTY TWENTY TWENTY TIME-ORDERED SPLIT
# ===============================================================

TRAIN_SIZE = int(np.floor(0.60 * N_EFFECTIVE))
VAL_SIZE = int(np.floor(0.20 * N_EFFECTIVE))
TEST_SIZE = int(N_EFFECTIVE - TRAIN_SIZE - VAL_SIZE)

# ==================================================================
# DEFINE POSITIONAL BOUNDARIES FOR TRAIN VALIDATION AND TEST WINDOWS
# ==================================================================

TRAIN_START_IDX = 0
TRAIN_END_IDX = TRAIN_START_IDX + TRAIN_SIZE - 1
VAL_START_IDX = TRAIN_END_IDX + 1
VAL_END_IDX = VAL_START_IDX + VAL_SIZE - 1
TEST_START_IDX = VAL_END_IDX + 1
TEST_END_IDX = N_EFFECTIVE - 1

# ========================================================
# EXTRACT DATE BOUNDARIES FOR AUDITABLE SPLIT PRINT OUTPUT
# ========================================================

TRAIN_START_DATE = CAUSAL_X_DF.index[TRAIN_START_IDX]
TRAIN_END_DATE = CAUSAL_X_DF.index[TRAIN_END_IDX]
VAL_START_DATE = CAUSAL_X_DF.index[VAL_START_IDX]
VAL_END_DATE = CAUSAL_X_DF.index[VAL_END_IDX]
TEST_START_DATE = CAUSAL_X_DF.index[TEST_START_IDX]
TEST_END_DATE = CAUSAL_X_DF.index[TEST_END_IDX]

# =================================================================
# BUILD EXPANDING-WINDOW BLOCK START INDICES USING TWENTY-DAY STEPS
# =================================================================

VAL_BLOCK_START_IDXS = list(range(VAL_START_IDX, TEST_START_IDX, WALK_FORWARD_STEP_DAYS))
TEST_BLOCK_START_IDXS = list(range(TEST_START_IDX, N_EFFECTIVE, WALK_FORWARD_STEP_DAYS))

# ======================================================================
# DEFINE LABEL MATURITY RULE THAT PROTECTS THE TWENTY-DAY FORWARD TARGET
# ======================================================================

def training_end_index_for_block(block_start_idx: int, horizon_days: int) -> int:
    # ===========================================================================
    # RETURN THE LAST INDEX WITH A MATURE LABEL AVAILABLE AT THE BLOCK START DATE
    # ===========================================================================

    return int(block_start_idx - horizon_days)

# =======================================================================
# BUILD SPLIT SUMMARY TABLE FOR NOTEBOOK 04 TRACEABILITY AND REPORT REUSE
# =======================================================================

SPLIT_SUMMARY_DF = pd.DataFrame(
    [
        {"split": "TRAIN", "start_date": str(TRAIN_START_DATE.date()), "end_date": str(TRAIN_END_DATE.date()), "row_count": TRAIN_SIZE},
        {"split": "VALIDATION", "start_date": str(VAL_START_DATE.date()), "end_date": str(VAL_END_DATE.date()), "row_count": VAL_SIZE},
        {"split": "TEST", "start_date": str(TEST_START_DATE.date()), "end_date": str(TEST_END_DATE.date()), "row_count": TEST_SIZE},
    ]
)

# ==========================================================================
# SAVE SPLIT SUMMARY TABLE FOR DOWNSTREAM NOTEBOOK 05 AND REPORT REFERENCING
# ==========================================================================

save_dataframe_csv(SPLIT_SUMMARY_DF, NB04_SPLIT_SUMMARY_PATH)

# =========================================================================
# PRINT SPLIT COUNTS BLOCK COUNTS AND SHAPES FOR NOTEBOOK VALIDATION OUTPUT
# =========================================================================

print("N_EFFECTIVE:", N_EFFECTIVE)
print("CAUSAL_X_DF_SHAPE:", CAUSAL_X_DF.shape)
print("TRAIN_SIZE:", TRAIN_SIZE, "|", "TRAIN_START_DATE:", str(TRAIN_START_DATE.date()), "|", "TRAIN_END_DATE:", str(TRAIN_END_DATE.date()))
print("VAL_SIZE:", VAL_SIZE, "|", "VAL_START_DATE:", str(VAL_START_DATE.date()), "|", "VAL_END_DATE:", str(VAL_END_DATE.date()))
print("TEST_SIZE:", TEST_SIZE, "|", "TEST_START_DATE:", str(TEST_START_DATE.date()), "|", "TEST_END_DATE:", str(TEST_END_DATE.date()))
print("VAL_BLOCK_COUNT:", len(VAL_BLOCK_START_IDXS))
print("TEST_BLOCK_COUNT:", len(TEST_BLOCK_START_IDXS))
print("NB04_SPLIT_SUMMARY_PATH:", str(NB04_SPLIT_SUMMARY_PATH))

# =========================================================================
# DISPLAY SPLIT SUMMARY AND THE FIRST FEW VALIDATION TEST BLOCK START DATES
# =========================================================================

display(SPLIT_SUMMARY_DF)
display(pd.DataFrame({"val_block_start_dates": [CAUSAL_X_DF.index[idx] for idx in VAL_BLOCK_START_IDXS[:5]]}))
display(pd.DataFrame({"test_block_start_dates": [CAUSAL_X_DF.index[idx] for idx in TEST_BLOCK_START_IDXS[:5]]}))

N_EFFECTIVE: 2496
CAUSAL_X_DF_SHAPE: (2496, 74)
TRAIN_SIZE: 1497 | TRAIN_START_DATE: 2016-01-04 | TRAIN_END_DATE: 2021-12-28
VAL_SIZE: 499 | VAL_START_DATE: 2021-12-29 | VAL_END_DATE: 2023-12-27
TEST_SIZE: 500 | TEST_START_DATE: 2023-12-28 | TEST_END_DATE: 2025-12-31
VAL_BLOCK_COUNT: 25
TEST_BLOCK_COUNT: 25
NB04_SPLIT_SUMMARY_PATH: /content/riskml-capstone/reports/tables/notebook04_split_summary.csv


,split,start_date,end_date,row_count
0,TRAIN,2016-01-04,2021-12-28,1497
1,VALIDATION,2021-12-29,2023-12-27,499
2,TEST,2023-12-28,2025-12-31,500


,val_block_start_dates
0,2021-12-29
1,2022-01-27
2,2022-02-25
3,2022-03-25
4,2022-04-25


,test_block_start_dates
0,2023-12-28
1,2024-01-29
2,2024-02-27
3,2024-03-26
4,2024-04-24


### 📌 Observations and Insights for CODE_BLOCK_E

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Time-ordered 60/20/20 split produces 1,497 train + 499 validation + 500 test = 2,496 total rows. Walk-forward block counts (25 validation, 25 test) match the Notebook 03 protocol exactly.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Computed the 60/20/20 time-ordered split on the 2,496-row effective modeling window: Train = 1,497 rows, Validation = 499 rows, Test = 500 rows.
- Extracted date boundaries for each split: Train (2016-01-04 → 2021-12-28), Validation (2021-12-29 → 2023-12-27), Test (2023-12-28 → 2025-12-31).
- Built expanding-window block start indices using 20-day steps: 25 validation blocks and 25 test blocks.
- Defined the label-safe boundary function `training_end_index_for_block()`, which truncates training data at `block_start − 20` to prevent forward-target leakage from overlapping 20-day windows.
- Saved `notebook04_split_summary.csv` to `reports/tables/` for downstream notebook and report referencing.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `N_EFFECTIVE`: 2,496
- `CAUSAL_X_DF_SHAPE`: (2,496, 74)
- `TRAIN`: 1,497 rows (2016-01-04 → 2021-12-28)
- `VALIDATION`: 499 rows (2021-12-29 → 2023-12-27)
- `TEST`: 500 rows (2023-12-28 → 2025-12-31)
- `VAL_BLOCK_COUNT`: 25
- `TEST_BLOCK_COUNT`: 25
- Saved artifact: `notebook04_split_summary.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The split sizes (1,497 + 499 + 500 = 2,496) account for every row in the effective modeling window with no gaps or overlaps.
- <span style="color: darkgreen;"><strong>✓</strong></span> The 25 test blocks × 20 days = 500 rows, matching the test split size exactly — confirming that no test observations are dropped by the block-stepping logic.
- <span style="color: darkgreen;"><strong>✓</strong></span> The test window (2023-12-28 → 2025-12-31) covers approximately two years of recent market data including the 2024 rate-cut cycle and Q4 2024 equity volatility — a meaningful out-of-sample period for evaluating forecast robustness.
- <span style="color: darkorange;"><strong>⚠</strong></span> The `CAUSAL_X_DF_SHAPE` of (2,496, 74) confirms that CODE_BLOCK_F will train on 74 features — exactly half the 151 features available to the Notebook 03 unconstrained baseline. This 51% reduction is the DAG constraint's primary mechanism, and any performance comparison in CODE_BLOCK_G must be interpreted in the context of this feature-set asymmetry.
- <span style="color: darkgreen;"><strong>✓</strong></span> The label-safe boundary function ensures that at each test block, training data ends 20 days before the block start — preventing the overlapping forward-volatility targets from contaminating the training set.
</div>

**Next step:** CODE_BLOCK_F trains the DAG-constrained XGBoost model using frozen hyperparameters from Notebook 03, runs walk-forward expanding-window evaluation over all 25 test blocks for all 14 tickers, and computes per-ticker RMSE and MAE.


***

In [18]:
# ============
# CODE_BLOCK_F
# ============

# ======================================================================
# FREEZE THE NOTEBOOK 03 XGBOOST HYPERPARAMETERS FOR FAIR DAG COMPARISON
# ======================================================================

FROZEN_XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_lambda": 1.0,
    "tree_method": "hist",
}

# ============================================================================
# DEFINE XGBOOST MODEL FACTORY WITH FROZEN PARAMETERS AND REPRODUCIBLE SEEDING
# ============================================================================

def build_xgb_regressor(params: Dict[str, float]) -> XGBRegressor:
    # ==========================================================================
    # RETURN A SQUARED-ERROR XGBOOST REGRESSOR FOR CONTINUOUS VOLATILITY TARGETS
    # ==========================================================================

    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=int(params["n_estimators"]),
        max_depth=int(params["max_depth"]),
        learning_rate=float(params["learning_rate"]),
        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),
        reg_lambda=float(params.get("reg_lambda", 1.0)),
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method=str(params.get("tree_method", "hist")),
    )

# =================================================================================
# DEFINE WALK-FORWARD PREDICTION FUNCTION USING THE NOTEBOOK 03 LABEL-SAFE PROTOCOL
# =================================================================================

def walk_forward_predict_xgb(
    x_df: pd.DataFrame,
    y_series: pd.Series,
    block_start_idxs: List[int],
    horizon_days: int,
    step_days: int,
    model_params: Dict[str, float],
) -> pd.DataFrame:
    # ========================================================
    # INITIALIZE ROW CONTAINER FOR LONG-FORM PREDICTION OUTPUT
    # ========================================================

    rows: List[Dict[str, float]] = []

    # =====================================================================
    # LOOP OVER TEST BLOCK START INDICES TO EMULATE EXPANDING-WINDOW REFITS
    # =====================================================================

    for block_start_idx in block_start_idxs:
        # ===========================================================
        # COMPUTE THE LAST TRAINING INDEX WITH A MATURE FORWARD LABEL
        # ===========================================================

        train_end_idx = training_end_index_for_block(block_start_idx, horizon_days)

        # ======================================================
        # SKIP THE BLOCK WHEN TOO LITTLE TRAINING HISTORY EXISTS
        # ======================================================

        if train_end_idx < 100:
            continue

        # ==================================================
        # SLICE TRAINING DATA UP TO THE LABEL-SAFE END INDEX
        # ==================================================

        x_train = x_df.iloc[: train_end_idx + 1]
        y_train = y_series.iloc[: train_end_idx + 1]

        # ============================================================
        # SLICE THE NEXT STEP-SIZED BLOCK FOR OUT-OF-SAMPLE PREDICTION
        # ============================================================

        block_end_idx_exclusive = min(block_start_idx + step_days, len(x_df))
        x_block = x_df.iloc[block_start_idx:block_end_idx_exclusive]
        y_block = y_series.iloc[block_start_idx:block_end_idx_exclusive]

        # =========================================
        # FIT A FRESH MODEL ON THE EXPANDING WINDOW
        # =========================================

        model = build_xgb_regressor(model_params)
        model.fit(x_train, y_train)

        # ========================================================
        # GENERATE PREDICTIONS FOR THE CURRENT OUT-OF-SAMPLE BLOCK
        # ========================================================

        y_pred = model.predict(x_block)

        # ===============================================================
        # APPEND ONE LONG-FORM ROW PER DATE FOR CLEAN EXPORT AND PLOTTING
        # ===============================================================

        for row_idx, dt in enumerate(x_block.index):
            rows.append(
                {
                    "date": str(pd.to_datetime(dt).date()),
                    "y_true": float(y_block.iloc[row_idx]),
                    "y_pred": float(y_pred[row_idx]),
                }
            )

    # =========================================
    # RETURN THE LONG-FORM PREDICTION DATAFRAME
    # =========================================

    return pd.DataFrame(rows)

# =====================================================================
# INITIALIZE METRIC AND PREDICTION CONTAINERS FOR THE CAUSAL MODEL LOOP
# =====================================================================

CAUSAL_METRIC_ROWS: List[Dict[str, float]] = []
CAUSAL_PREDICTION_ROWS: List[Dict[str, float]] = []

# ================================================================================
# LOOP OVER ALL TICKERS TO FIT THE DAG-CONSTRAINED MODEL AND SCORE THE TEST WINDOW
# ================================================================================

for ticker in TICKER_LIST:
    # =============================================================
    # SELECT THE GATED CAUSAL FEATURE MATRIX FOR THE CURRENT TICKER
    # =============================================================

    x_df = CAUSAL_X_DF

    # =================================================================
    # SELECT THE TARGET SERIES THROUGH THE TARGET COLUMN GOVERNANCE MAP
    # =================================================================

    y_series = EFF_TARGET_DF[TARGET_COL_MAP[ticker]]

    # ===========================================================
    # RUN LABEL-SAFE WALK-FORWARD PREDICTION OVER THE TEST BLOCKS
    # ===========================================================

    preds_df = walk_forward_predict_xgb(
        x_df=x_df,
        y_series=y_series,
        block_start_idxs=TEST_BLOCK_START_IDXS,
        horizon_days=FORECAST_HORIZON_DAYS,
        step_days=WALK_FORWARD_STEP_DAYS,
        model_params=FROZEN_XGB_PARAMS,
    )

    # ======================================================================
    # RAISE AN ERROR WHEN NO PREDICTIONS ARE PRODUCED FOR THE CURRENT TICKER
    # ======================================================================

    if preds_df.empty:
        raise ValueError(f"NO CAUSAL PREDICTIONS PRODUCED FOR TICKER={ticker}")

    # ===========================================
    # COMPUTE RMSE AND MAE FOR THE CURRENT TICKER
    # ===========================================

    rmse_val = compute_rmse(preds_df["y_true"].values, preds_df["y_pred"].values)
    mae_val = float(mean_absolute_error(preds_df["y_true"].values, preds_df["y_pred"].values))

    # ==========================================================
    # APPEND PER-TICKER METRICS FOR THE NOTEBOOK 04 METRIC TABLE
    # ==========================================================

    CAUSAL_METRIC_ROWS.append(
        {
            "model": "CAUSAL_XGBOOST",
            "ticker": ticker,
            "rmse": rmse_val,
            "mae": mae_val,
            "n_obs": int(len(preds_df)),
            "feature_count": int(x_df.shape[1]),
        }
    )

    # ======================================================
    # APPEND LONG-FORM PREDICTIONS FOR EXPORT AND PLOT REUSE
    # ======================================================

    for row in preds_df.to_dict(orient="records"):
        CAUSAL_PREDICTION_ROWS.append(
            {
                "date": row["date"],
                "ticker": ticker,
                "model": "CAUSAL_XGBOOST",
                "y_true": row["y_true"],
                "y_pred": row["y_pred"],
            }
        )

# ===========================================================================
# BUILD THE CAUSAL METRICS DATAFRAME AND APPEND AN EQUAL-WEIGHT AGGREGATE ROW
# ===========================================================================

CAUSAL_METRICS_DF = pd.DataFrame(CAUSAL_METRIC_ROWS)

CAUSAL_AGG_ROW = {
    "model": "CAUSAL_XGBOOST",
    "ticker": "AGGREGATE_EQUAL_WEIGHT",
    "rmse": float(CAUSAL_METRICS_DF["rmse"].mean()),
    "mae": float(CAUSAL_METRICS_DF["mae"].mean()),
    "n_obs": int(CAUSAL_METRICS_DF["n_obs"].min()),
    "feature_count": int(CAUSAL_METRICS_DF["feature_count"].max()),
}

CAUSAL_METRICS_DF = pd.concat([CAUSAL_METRICS_DF, pd.DataFrame([CAUSAL_AGG_ROW])], ignore_index=True)

# =====================================================================
# BUILD THE LONG-FORM CAUSAL PREDICTION DATAFRAME AND SAVE BOTH OUTPUTS
# =====================================================================

CAUSAL_PREDICTIONS_LONG_DF = pd.DataFrame(CAUSAL_PREDICTION_ROWS)
save_dataframe_csv(CAUSAL_METRICS_DF, NB04_CAUSAL_METRICS_PATH)
save_dataframe_csv(CAUSAL_PREDICTIONS_LONG_DF, NB04_CAUSAL_PREDICTIONS_TABLE_PATH)
save_dataframe_csv(CAUSAL_PREDICTIONS_LONG_DF, NB04_CAUSAL_PREDICTIONS_DATA_PATH)

# ============================================================================
# PRINT THE FROZEN PARAMETER SET OUTPUT PATHS AND THE CAUSAL AGGREGATE METRICS
# ============================================================================

print("FROZEN_XGB_PARAMS:", json.dumps(FROZEN_XGB_PARAMS, indent=2))
print("CAUSAL_FEATURE_COUNT:", CAUSAL_X_DF.shape[1])
print("NB04_CAUSAL_METRICS_PATH:", str(NB04_CAUSAL_METRICS_PATH))
print("NB04_CAUSAL_PREDICTIONS_TABLE_PATH:", str(NB04_CAUSAL_PREDICTIONS_TABLE_PATH))
print("NB04_CAUSAL_PREDICTIONS_DATA_PATH:", str(NB04_CAUSAL_PREDICTIONS_DATA_PATH))

# =================================================================
# DISPLAY THE CAUSAL METRIC TABLE AND A LONG-FORM PREDICTION SAMPLE
# =================================================================

display(CAUSAL_METRICS_DF.sort_values(["ticker"]).reset_index(drop=True))
display(CAUSAL_PREDICTIONS_LONG_DF.head(20))

FROZEN_XGB_PARAMS: {
  "n_estimators": 400,
  "max_depth": 4,
  "learning_rate": 0.05,
  "subsample": 0.8,
  "colsample_bytree": 0.8,
  "reg_lambda": 1.0,
  "tree_method": "hist"
}
CAUSAL_FEATURE_COUNT: 74
NB04_CAUSAL_METRICS_PATH: /content/riskml-capstone/reports/tables/notebook04_causal_rmse_mae.csv
NB04_CAUSAL_PREDICTIONS_TABLE_PATH: /content/riskml-capstone/reports/tables/notebook04_causal_predictions_long.csv
NB04_CAUSAL_PREDICTIONS_DATA_PATH: /content/riskml-capstone/data/processed/notebook04_causal_predictions_long.csv


,model,ticker,rmse,mae,n_obs,feature_count
0,CAUSAL_XGBOOST,AGGREGATE_EQUAL_WEIGHT,0.081112,0.050080,500,74
1,CAUSAL_XGBOOST,DBC,0.044915,0.035933,500,74
2,CAUSAL_XGBOOST,EEM,0.080995,0.052077,500,74
3,CAUSAL_XGBOOST,EFA,0.086577,0.047173,500,74
4,CAUSAL_XGBOOST,GLD,0.063799,0.044103,500,74
5,CAUSAL_XGBOOST,HYG,0.044356,0.025161,500,74
6,CAUSAL_XGBOOST,IWM,0.092615,0.054303,500,74
7,CAUSAL_XGBOOST,LQD,0.042122,0.022042,500,74
8,CAUSAL_XGBOOST,QQQ,0.100368,0.066316,500,74
9,CAUSAL_XGBOOST,SPY,0.098160,0.060865,500,74


,date,ticker,model,y_true,y_pred
0,2023-12-28,SPY,CAUSAL_XGBOOST,0.095915,0.124487
1,2023-12-29,SPY,CAUSAL_XGBOOST,0.094912,0.114656
2,2024-01-02,SPY,CAUSAL_XGBOOST,0.112099,0.110453
3,2024-01-03,SPY,CAUSAL_XGBOOST,0.114090,0.108563
4,2024-01-04,SPY,CAUSAL_XGBOOST,0.115906,0.109790
5,2024-01-05,SPY,CAUSAL_XGBOOST,0.118084,0.108847
6,2024-01-08,SPY,CAUSAL_XGBOOST,0.109878,0.108035
7,2024-01-09,SPY,CAUSAL_XGBOOST,0.111230,0.104641
8,2024-01-10,SPY,CAUSAL_XGBOOST,0.110809,0.105370
9,2024-01-11,SPY,CAUSAL_XGBOOST,0.111012,0.104551


### 📌 Observations and Insights for CODE_BLOCK_F

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — All 14 tickers produced 500 out-of-sample predictions each across 25 walk-forward test blocks. Three artifact files saved successfully. Aggregate RMSE = 0.0811, aggregate MAE = 0.0501.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Froze the Notebook 03 XGBoost hyperparameters (n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.80, colsample_bytree=0.80, reg_lambda=1.0, tree_method=hist) to ensure that any performance difference relative to baselines is attributable to the feature restriction, not hyperparameter variation.
- Defined the `build_xgb_regressor()` factory function, which constructs a `reg:squarederror` XGBoost model with frozen parameters and `random_state=692`.
- Defined the `walk_forward_predict_xgb()` function implementing the label-safe expanding-window protocol: at each of 25 test blocks, training data is truncated at `block_start − 20` to prevent forward-target leakage, a fresh model is fit on the expanding window, and predictions are generated for the next 20-day block.
- Looped over all 14 tickers, routing each target through `TARGET_COL_MAP` (Target-Column Governance Rule), and accumulated per-ticker RMSE, MAE, observation count, and feature count.
- Appended an `AGGREGATE_EQUAL_WEIGHT` row computing the mean RMSE and mean MAE across 14 tickers.
- Saved three output artifacts: `notebook04_causal_rmse_mae.csv` (metrics), `notebook04_causal_predictions_long.csv` (to `reports/tables/`), and a duplicate copy to `data/processed/` for downstream Notebook 05 and Notebook 06 consumption.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `CAUSAL_FEATURE_COUNT`: 74 (DAG-constrained)
- `n_obs` per ticker: 500 (25 blocks × 20 days)
- Aggregate RMSE: **0.0811** | Aggregate MAE: **0.0501**
- Lowest RMSE: LQD (0.0421) | Highest RMSE: XLE (0.1378)
- Saved artifacts: `notebook04_causal_rmse_mae.csv`, `notebook04_causal_predictions_long.csv` (×2 locations)
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> All 14 tickers produced exactly 500 predictions — no blocks were skipped due to insufficient training history, confirming that the label-safe boundary `block_start − 20` always exceeds the 100-observation minimum.
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The constrained aggregate RMSE (0.0811) will be compared against the Notebook 03 unconstrained XGBoost aggregate RMSE (0.0698) and EWMA aggregate RMSE (0.0708) in CODE_BLOCK_G. The constrained model uses only 74 features versus 151 in the unconstrained baseline — a 51% feature reduction — so some RMSE increase is expected for equity tickers where momentum cross-contamination provided useful signal.
- <span style="color: darkorange;"><strong>⚠</strong></span> Fixed-income tickers (TLT=0.0544, LQD=0.0421, HYG=0.0444) show relatively low absolute RMSE values. The key question for CODE_BLOCK_G is whether these represent an improvement over the Notebook 03 baselines, where unconstrained XGBoost overfit on noise features for low-volatility fixed-income assets — the strongest a priori hypothesis from the handoff (Section 5.5b).
- <span style="color: darkgreen;"><strong>✓</strong></span> The long-form prediction sample (first 20 rows for SPY) shows the constrained model tracking the general level of realized volatility (y_true ≈ 0.10–0.14, y_pred ≈ 0.10–0.12) during early 2024, with the model slightly underforecasting — a pattern consistent with volatility mean-reversion bias in tree-based models.
- <span style="color: darkgreen;"><strong>✓</strong></span> The prediction DataFrame includes the `model` column tagged as `CAUSAL_XGBOOST`, enabling clean concatenation with the Notebook 03 `EWMA` and `XGBOOST` predictions in CODE_BLOCK_J for the combined forecast overlay figure.
</div>

**Next step:** CODE_BLOCK_G merges the constrained metrics against the Notebook 03 EWMA and XGBoost baselines, computes per-ticker RMSE and MAE deltas, and counts wins versus both baselines — the primary H1 test.


***

In [ ]:
import os
from getpass import getpass

os.chdir("/content/riskml-capstone")

!git config user.email "steve@youremail.com"
!git config user.name "stevearchuleta"

token = getpass("GitHub PAT: ")
!git remote set-url origin https://{token}@github.com/stevearchuleta/riskml-capstone.git

!git add -A
!git commit -m "NB04: CODE_BLOCKs A-F complete with markdown observation cells, causal predictions saved"
!git push origin main

GitHub PAT: ··········
[main 14c1ee2] NB04: CODE_BLOCKs A-F complete with markdown observation cells, causal predictions saved
 5 files changed, 7175 insertions(+)
 create mode 100644 reports/tables/notebook04_causal_predictions_long.csv
 create mode 100644 reports/tables/notebook04_causal_rmse_mae.csv
 create mode 100644 reports/tables/notebook04_dag_gating_summary.csv
 create mode 100644 reports/tables/notebook04_effective_window_summary.csv
 create mode 100644 reports/tables/notebook04_split_summary.csv
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 153.09 KiB | 4.94 MiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/stevearchuleta/riskml-capstone.git
   827428e..14c1ee2  main -> main


In [19]:
# ============
# CODE_BLOCK_G
# ============

# ================================================================================
# DEFINE HELPER TO RECOMPUTE METRICS FROM A LONG-FORM PREDICTION TABLE WHEN NEEDED
# ================================================================================

def metrics_from_long_predictions(long_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    # ====================================================================
    # FILTER TO ONE MODEL AND ASSERT THAT THE LONG-FORM TABLE IS NOT EMPTY
    # ====================================================================

    model_df = long_df[long_df["model"] == model_name].copy()

    if model_df.empty:
        raise ValueError(f"LONG-FORM PREDICTION TABLE CONTAINS NO ROWS FOR MODEL={model_name}")

    # ==========================================================
    # GROUP BY TICKER AND COMPUTE RMSE MAE AND OBSERVATION COUNT
    # ==========================================================

    rows = []
    for ticker, ticker_df in model_df.groupby("ticker", sort=True):
        rows.append(
            {
                "model": model_name,
                "ticker": ticker,
                "rmse": compute_rmse(ticker_df["y_true"].values, ticker_df["y_pred"].values),
                "mae": float(mean_absolute_error(ticker_df["y_true"].values, ticker_df["y_pred"].values)),
                "n_obs": int(len(ticker_df)),
            }
        )

    # ====================================
    # APPEND AN EQUAL-WEIGHT AGGREGATE ROW
    # ====================================

    out_df = pd.DataFrame(rows)
    agg_row = {
        "model": model_name,
        "ticker": "AGGREGATE_EQUAL_WEIGHT",
        "rmse": float(out_df["rmse"].mean()),
        "mae": float(out_df["mae"].mean()),
        "n_obs": int(out_df["n_obs"].min()),
    }

    # =======================================================
    # RETURN THE METRIC TABLE WITH THE AGGREGATE ROW APPENDED
    # =======================================================

    return pd.concat([out_df, pd.DataFrame([agg_row])], ignore_index=True)

# ==============================================================================
# LOAD NOTEBOOK 03 METRICS WHEN AVAILABLE OR RECOMPUTE FROM BASELINE PREDICTIONS
# ==============================================================================

if NB03_BASELINE_METRICS_DF is not None:
    BASELINE_METRICS_DF = NB03_BASELINE_METRICS_DF.copy()
elif NB03_BASELINE_PREDICTIONS_DF is not None:
    EWMA_FROM_LONG_DF = metrics_from_long_predictions(NB03_BASELINE_PREDICTIONS_DF, "EWMA")
    XGBOOST_FROM_LONG_DF = metrics_from_long_predictions(NB03_BASELINE_PREDICTIONS_DF, "XGBOOST")
    BASELINE_METRICS_DF = pd.concat([EWMA_FROM_LONG_DF, XGBOOST_FROM_LONG_DF], ignore_index=True)
else:
    raise FileNotFoundError("NOTEBOOK 03 BASELINE METRICS OR PREDICTIONS ARTIFACT REQUIRED BEFORE NOTEBOOK 04 COMPARISON")

# =========================================================================
# SUBSET THE BASELINE TABLE TO THE TWO REFERENCE MODELS USED IN NOTEBOOK 04
# =========================================================================

BASELINE_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"].isin(["EWMA", "XGBOOST"])].copy()

# =========================================================
# BUILD XGBOOST AND EWMA REFERENCE TABLES FOR CLEAN MERGING
# =========================================================

BASELINE_XGB_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"] == "XGBOOST"].copy()
BASELINE_EWMA_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"] == "EWMA"].copy()

# ============================================================================
# MERGE CAUSAL METRICS AGAINST THE XGBOOST BASELINE FOR THE PRIMARY H ONE TEST
# ============================================================================

CAUSAL_VS_XGB_DF = (
    CAUSAL_METRICS_DF.rename(columns={"rmse": "causal_rmse", "mae": "causal_mae", "n_obs": "causal_n_obs"})
    .merge(
        BASELINE_XGB_METRICS_DF.rename(columns={"rmse": "xgboost_rmse", "mae": "xgboost_mae", "n_obs": "xgboost_n_obs"}).drop(columns=["model"]),
        on="ticker",
        how="left",
    )
    .assign(
        causal_minus_xgboost_rmse=lambda df: df["causal_rmse"] - df["xgboost_rmse"],
        causal_minus_xgboost_mae=lambda df: df["causal_mae"] - df["xgboost_mae"],
        rmse_improves_vs_xgboost=lambda df: df["causal_minus_xgboost_rmse"] < 0,
        mae_improves_vs_xgboost=lambda df: df["causal_minus_xgboost_mae"] < 0,
    )
)

# =========================================================
# ADD EWMA REFERENCE COLUMNS FOR SECONDARY BASELINE CONTEXT
# =========================================================

CAUSAL_VS_BASELINE_DF = (
    CAUSAL_VS_XGB_DF.merge(
        BASELINE_EWMA_METRICS_DF.rename(columns={"rmse": "ewma_rmse", "mae": "ewma_mae", "n_obs": "ewma_n_obs"}).drop(columns=["model"]),
        on="ticker",
        how="left",
    )
    .assign(
        causal_minus_ewma_rmse=lambda df: df["causal_rmse"] - df["ewma_rmse"],
        causal_minus_ewma_mae=lambda df: df["causal_mae"] - df["ewma_mae"],
        rmse_improves_vs_ewma=lambda df: df["causal_minus_ewma_rmse"] < 0,
        mae_improves_vs_ewma=lambda df: df["causal_minus_ewma_mae"] < 0,
    )
)

# ====================================================================
# SAVE THE NOTEBOOK 04 PRIMARY COMPARISON TABLE FOR REPORT INTEGRATION
# ====================================================================

save_dataframe_csv(CAUSAL_VS_BASELINE_DF, NB04_COMPARISON_PATH)

# ====================================================================
# COMPUTE SIMPLE WIN COUNTS AGAINST BOTH BASELINES FOR NOTEBOOK OUTPUT
# ====================================================================

NON_AGG_COMPARISON_DF = CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"].copy()
RMSE_WINS_VS_XGB = int(NON_AGG_COMPARISON_DF["rmse_improves_vs_xgboost"].sum())
RMSE_WINS_VS_EWMA = int(NON_AGG_COMPARISON_DF["rmse_improves_vs_ewma"].sum())

# =======================================================================
# PRINT COMPARISON PATH AND WIN COUNTS FOR NOTEBOOK 04 INTERPRETIVE CELLS
# =======================================================================

print("NB04_COMPARISON_PATH:", str(NB04_COMPARISON_PATH))
print("RMSE_WINS_VS_XGBOOST:", RMSE_WINS_VS_XGB)
print("RMSE_WINS_VS_EWMA:", RMSE_WINS_VS_EWMA)

# ===============================================================
# DISPLAY THE FULL CAUSAL VERSUS BASELINE METRIC COMPARISON TABLE
# ===============================================================

display(CAUSAL_VS_BASELINE_DF.sort_values(["ticker"]).reset_index(drop=True))

NB04_COMPARISON_PATH: /content/riskml-capstone/reports/tables/causal_vs_baseline_rmse_mae.csv
RMSE_WINS_VS_XGBOOST: 0
RMSE_WINS_VS_EWMA: 5


,model,ticker,causal_rmse,causal_mae,causal_n_obs,feature_count,xgboost_rmse,xgboost_mae,xgboost_n_obs,causal_minus_xgboost_rmse,causal_minus_xgboost_mae,rmse_improves_vs_xgboost,mae_improves_vs_xgboost,ewma_rmse,ewma_mae,ewma_n_obs,causal_minus_ewma_rmse,causal_minus_ewma_mae,rmse_improves_vs_ewma,mae_improves_vs_ewma
0,CAUSAL_XGBOOST,AGGREGATE_EQUAL_WEIGHT,0.081112,0.050080,500,74,0.069814,0.049100,500,0.011298,0.000980,False,False,0.070824,0.048309,500,0.010288,0.001772,False,False
1,CAUSAL_XGBOOST,DBC,0.044915,0.035933,500,74,0.040213,0.031392,500,0.004702,0.004541,False,False,0.048130,0.036194,500,-0.003215,-0.000261,True,True
2,CAUSAL_XGBOOST,EEM,0.080995,0.052077,500,74,0.068626,0.052573,500,0.012369,-0.000496,False,True,0.071634,0.051192,500,0.009361,0.000885,False,False
3,CAUSAL_XGBOOST,EFA,0.086577,0.047173,500,74,0.068190,0.044796,500,0.018388,0.002377,False,False,0.072764,0.043087,500,0.013813,0.004087,False,False
4,CAUSAL_XGBOOST,GLD,0.063799,0.044103,500,74,0.063428,0.045630,500,0.000372,-0.001526,False,True,0.068266,0.049558,500,-0.004467,-0.005455,True,True
5,CAUSAL_XGBOOST,HYG,0.044356,0.025161,500,74,0.033317,0.020632,500,0.011039,0.004529,False,False,0.027034,0.017535,500,0.017323,0.007626,False,False
6,CAUSAL_XGBOOST,IWM,0.092615,0.054303,500,74,0.081688,0.058732,500,0.010927,-0.004430,False,True,0.083909,0.062949,500,0.008706,-0.008646,False,True
7,CAUSAL_XGBOOST,LQD,0.042122,0.022042,500,74,0.033319,0.020265,500,0.008803,0.001777,False,False,0.021307,0.015117,500,0.020816,0.006925,False,False
8,CAUSAL_XGBOOST,QQQ,0.100368,0.066316,500,74,0.095097,0.067822,500,0.005271,-0.001506,False,True,0.108177,0.074471,500,-0.007809,-0.008155,True,True
9,CAUSAL_XGBOOST,SPY,0.098160,0.060865,500,74,0.089175,0.058166,500,0.008986,0.002699,False,False,0.099287,0.064059,500,-0.001127,-0.003195,True,True


### 📌 Observations and Insights for CODE_BLOCK_G

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Causal metrics merged successfully against both Notebook 03 baselines (EWMA and XGBoost) for all 14 tickers plus the aggregate row. Comparison table saved to artifact.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Loaded Notebook 03 baseline metrics from the saved `notebook03_baseline_rmse_mae.csv` artifact (primary path) with a fallback to recompute from long-form predictions if the metrics CSV were unavailable.
- Split baseline metrics into separate EWMA and XGBoost reference tables for clean merging.
- Merged causal RMSE/MAE against the XGBoost baseline (primary H1 test), computing per-ticker `causal_minus_xgboost_rmse` and `causal_minus_xgboost_mae` deltas plus boolean improvement flags.
- Added EWMA reference columns with analogous delta and improvement flag columns for secondary baseline context.
- Saved the full comparison table to `causal_vs_baseline_rmse_mae.csv` in `reports/tables/`.
- Computed win counts: RMSE wins versus XGBoost and RMSE wins versus EWMA across 14 non-aggregate tickers.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `RMSE_WINS_VS_XGBOOST`: **0 out of 14** — the constrained model does not beat the unconstrained XGBoost on RMSE for any individual ticker
- `RMSE_WINS_VS_EWMA`: **5 out of 14** — the constrained model beats EWMA on DBC, GLD, QQQ, SPY, and XLK
- Aggregate causal RMSE: 0.0811 vs XGBoost 0.0698 (delta = +0.0113) vs EWMA 0.0708 (delta = +0.0103)
- Saved artifact: `causal_vs_baseline_rmse_mae.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The constrained model wins 0 of 14 RMSE comparisons against the unconstrained XGBoost. This result is consistent with the 51% feature reduction removing useful cross-node signal — the unconstrained model had access to momentum and value features that provided genuine predictive value for certain tickers. The DAG constraint prioritizes interpretability and structural coherence over raw accuracy.
- <span style="color: darkorange;"><strong>⚠</strong></span> The a priori hypothesis from the handoff (Section 5.5b) — that fixed-income tickers (TLT, LQD, HYG) would improve under DAG constraints because the unconstrained model overfits on noise features — is **not confirmed**. All three fixed-income tickers show positive RMSE deltas (TLT: +0.0117, LQD: +0.0088, HYG: +0.0110), indicating the constrained model performs worse, not better.
- <span style="color: darkgreen;"><strong>✓</strong></span> On MAE (the robust complement), the constrained model improves for 6 of 14 tickers versus XGBoost (EEM, GLD, IWM, QQQ, XLK, XLV), suggesting that the constrained model reduces median-level errors for some tickers even while increasing tail-error sensitivity (captured by RMSE).
- <span style="color: darkgreen;"><strong>✓</strong></span> The 5 RMSE wins versus EWMA (DBC, GLD, QQQ, SPY, XLK) demonstrate that the DAG-constrained model still outperforms the simple EWMA benchmark for key equity and commodity tickers — the constrained model provides value relative to a non-ML baseline.
- <span style="color: darkorange;"><strong>⚠</strong></span> The aggregate RMSE degradation of +0.0113 (16.2% relative to XGBoost baseline) is larger than the 2–3% threshold noted in the handoff (Section 5.5d) as still informative. The report narrative should frame the result as a meaningful accuracy-interpretability trade-off rather than a strict improvement.
</div>

**Next step:** CODE_BLOCK_H fits final constrained models on the full effective window for representative tickers (SPY, TLT, GLD) and extracts gain-based feature importance to confirm DAG prefix purity and analyze which volatility features drive the constrained forecasts.


***

In [20]:
# ============
# CODE_BLOCK_H
# ============

# ==================================================================
# SELECT REPRESENTATIVE TICKERS FOR FINAL INTERPRETABILITY ARTIFACTS
# ==================================================================

REP_TICKERS_IMPORTANCE = [ticker for ticker in ["SPY", "TLT", "GLD"] if ticker in TICKER_LIST]

if len(REP_TICKERS_IMPORTANCE) == 0:
    REP_TICKERS_IMPORTANCE = TICKER_LIST[:3]

# =========================================================================
# DEFINE HELPER TO CONVERT XGBOOST GAIN DICTIONARIES INTO SORTED DATAFRAMES
# =========================================================================

def gain_dict_to_dataframe(gain_dict: Dict[str, float], ticker: str, model_variant: str) -> pd.DataFrame:
    # ===========================================================================
    # BUILD A LONG-FORM IMPORTANCE TABLE WITH PREFIX TAGS AND WITHIN-TICKER RANKS
    # ===========================================================================

    fi_df = pd.DataFrame([{"feature": feature, "gain": float(gain)} for feature, gain in gain_dict.items()])

    if fi_df.empty:
        return pd.DataFrame(columns=["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"])

    fi_df = fi_df.sort_values("gain", ascending=False).reset_index(drop=True)
    fi_df["ticker"] = ticker
    fi_df["model_variant"] = model_variant
    fi_df["dag_prefix"] = fi_df["feature"].map(extract_feature_prefix)
    fi_df["gain_share"] = fi_df["gain"] / fi_df["gain"].sum()
    fi_df["rank_within_ticker"] = np.arange(1, len(fi_df) + 1)

    # =====================================
    # RETURN THE LONG-FORM IMPORTANCE TABLE
    # =====================================

    return fi_df[["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"]]

# ======================================================================================
# FIT FINAL CONSTRAINED MODELS ON THE FULL EFFECTIVE WINDOW FOR INTERPRETABILITY EXPORTS
# ======================================================================================

FINAL_CAUSAL_MODELS: Dict[str, XGBRegressor] = {}
CAUSAL_IMPORTANCE_DFS: List[pd.DataFrame] = []

for ticker in REP_TICKERS_IMPORTANCE:
    # =================================================================================
    # SELECT THE FULL GATED FEATURE MATRIX AND THE TARGET SERIES FOR THE CURRENT TICKER
    # =================================================================================

    x_full = CAUSAL_X_DF.copy()
    y_full = EFF_TARGET_DF[TARGET_COL_MAP[ticker]].copy()

    # ===============================
    # FIT THE FINAL CONSTRAINED MODEL
    # ===============================

    final_model = build_xgb_regressor(FROZEN_XGB_PARAMS)
    final_model.fit(x_full, y_full)
    FINAL_CAUSAL_MODELS[ticker] = final_model

    # =========================================================
    # SAVE THE FINAL CONSTRAINED MODEL JSON FOR REPRODUCIBILITY
    # =========================================================

    model_path = MODELS_DIR / f"notebook04_xgb_causal_{ticker}.json"
    final_model.save_model(str(model_path))

    # ============================================================
    # EXTRACT GAIN-BASED FEATURE IMPORTANCE FOR THE CURRENT TICKER
    # ============================================================

    gain_dict = final_model.get_booster().get_score(importance_type="gain")
    causal_fi_df = gain_dict_to_dataframe(gain_dict, ticker=ticker, model_variant="CAUSAL_CONSTRAINED")
    causal_fi_df["model_path"] = str(model_path)
    CAUSAL_IMPORTANCE_DFS.append(causal_fi_df)

# ========================================================================
# CONCATENATE THE CONSTRAINED IMPORTANCE TABLES AND VALIDATE PREFIX PURITY
# ========================================================================

CAUSAL_FEATURE_IMPORTANCE_DF = pd.concat(CAUSAL_IMPORTANCE_DFS, ignore_index=True)

FORBIDDEN_IMPORTANCE_ROWS_DF = CAUSAL_FEATURE_IMPORTANCE_DF[
    CAUSAL_FEATURE_IMPORTANCE_DF["dag_prefix"].isin([prefix.replace("__", "") for prefix in FORBIDDEN_PREFIXES_RISK])
].copy()

if not FORBIDDEN_IMPORTANCE_ROWS_DF.empty:
    raise ValueError("FORBIDDEN DAG PREFIX APPEARED IN CONSTRAINED FEATURE IMPORTANCE OUTPUT")

# =====================================================================
# SAVE THE CONSTRAINED FEATURE IMPORTANCE TABLE FOR THE REPORT APPENDIX
# =====================================================================

save_dataframe_csv(CAUSAL_FEATURE_IMPORTANCE_DF, NB04_FEATURE_IMPORTANCE_PATH)

# ===========================================================
# PRINT REPRESENTATIVE TICKERS AND MODEL EXPORT PATH EXAMPLES
# ===========================================================

print("REP_TICKERS_IMPORTANCE:", REP_TICKERS_IMPORTANCE)
print("NB04_FEATURE_IMPORTANCE_PATH:", str(NB04_FEATURE_IMPORTANCE_PATH))
print("FINAL_CAUSAL_MODEL_EXPORTS:", [str(MODELS_DIR / f"notebook04_xgb_causal_{ticker}.json") for ticker in REP_TICKERS_IMPORTANCE])

# ===================================================================
# DISPLAY THE TOP CONSTRAINED FEATURES FOR EACH REPRESENTATIVE TICKER
# ===================================================================

display(CAUSAL_FEATURE_IMPORTANCE_DF.groupby("ticker", group_keys=False).head(10).reset_index(drop=True))

REP_TICKERS_IMPORTANCE: ['SPY', 'TLT', 'GLD']
NB04_FEATURE_IMPORTANCE_PATH: /content/riskml-capstone/reports/tables/notebook04_causal_feature_importance.csv
FINAL_CAUSAL_MODEL_EXPORTS: ['/content/riskml-capstone/reports/models/notebook04_xgb_causal_SPY.json', '/content/riskml-capstone/reports/models/notebook04_xgb_causal_TLT.json', '/content/riskml-capstone/reports/models/notebook04_xgb_causal_GLD.json']


,ticker,model_variant,feature,dag_prefix,gain,gain_share,rank_within_ticker,model_path
0,SPY,CAUSAL_CONSTRAINED,VOL__QQQ__ewma_vol__span21,VOL,0.635768,0.205927,1,/content/riskml-capstone/reports/models/notebo...
1,SPY,CAUSAL_CONSTRAINED,VOL__XLV__rvol__10d,VOL,0.170143,0.055110,2,/content/riskml-capstone/reports/models/notebo...
2,SPY,CAUSAL_CONSTRAINED,MACRO__vixcls,MACRO,0.162709,0.052702,3,/content/riskml-capstone/reports/models/notebo...
3,SPY,CAUSAL_CONSTRAINED,VOL__DBC__ewma_vol__span21,VOL,0.131760,0.042678,4,/content/riskml-capstone/reports/models/notebo...
4,SPY,CAUSAL_CONSTRAINED,VOL__SPY__ewma_vol__span21,VOL,0.122220,0.039587,5,/content/riskml-capstone/reports/models/notebo...
5,SPY,CAUSAL_CONSTRAINED,VOL__EEM__rvol__21d,VOL,0.120266,0.038955,6,/content/riskml-capstone/reports/models/notebo...
6,SPY,CAUSAL_CONSTRAINED,VOL__XLK__rvol__21d,VOL,0.117894,0.038186,7,/content/riskml-capstone/reports/models/notebo...
7,SPY,CAUSAL_CONSTRAINED,VOL__XLK__ewma_vol__span21,VOL,0.100413,0.032524,8,/content/riskml-capstone/reports/models/notebo...
8,SPY,CAUSAL_CONSTRAINED,VOL__LQD__rvol__21d,VOL,0.087154,0.028229,9,/content/riskml-capstone/reports/models/notebo...
9,SPY,CAUSAL_CONSTRAINED,VOL__IWM__rvol__63d,VOL,0.081882,0.026522,10,/content/riskml-capstone/reports/models/notebo...


### 📌 Observations and Insights for CODE_BLOCK_H

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Final constrained models fit and exported for SPY, TLT, and GLD. Feature importance tables contain only VOL__, MACRO__, and REGIME__ prefixes — zero forbidden-prefix features detected. Three model JSON files and one importance CSV saved.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Selected three representative tickers (SPY, TLT, GLD) spanning US equity, fixed income, and commodity asset classes for interpretability analysis.
- Fit final constrained XGBoost models on the full 2,496-row effective window (not the walk-forward splits) for each representative ticker, using frozen hyperparameters.
- Saved serialized model JSON files to `reports/models/`: `notebook04_xgb_causal_SPY.json`, `notebook04_xgb_causal_TLT.json`, `notebook04_xgb_causal_GLD.json`.
- Extracted gain-based feature importance dictionaries from each model's booster and converted each to a long-form DataFrame with DAG prefix tags, gain shares, and within-ticker ranks.
- Validated that zero features from forbidden DAG prefixes (MOM, VAL, ML, SENT) appeared in the constrained importance output — confirming that the gating from CODE_BLOCK_D held through model fitting.
- Saved the concatenated importance table to `notebook04_causal_feature_importance.csv`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `REP_TICKERS_IMPORTANCE`: ['SPY', 'TLT', 'GLD']
- **SPY top feature**: `VOL__QQQ__ewma_vol__span21` (gain = 0.636, gain_share = 20.6%)
- **TLT top feature**: `VOL__LQD__rvol__63d` (gain = 0.113, gain_share = 13.3%)
- **GLD top feature**: `VOL__GLD__ewma_vol__span63` (gain = 0.069, gain_share = 9.3%)
- Saved artifacts: `notebook04_causal_feature_importance.csv`, 3 model JSON files
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The constrained SPY model's top feature (`VOL__QQQ__ewma_vol__span21`, 20.6% gain share) is a volatility-family feature from a closely correlated equity ETF — economically sensible because QQQ and SPY share substantial market-risk exposure.
- <span style="color: darkgreen;"><strong>✓</strong></span> The constrained TLT model relies heavily on credit-spread-related volatility features (`VOL__LQD__rvol__63d` at rank 1, `VOL__TLT__rvol__63d` at rank 2), reflecting the empirical connection between long-duration Treasury volatility and investment-grade credit volatility.
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The GLD model uniquely elevates `REGIME__vix_high` to rank 2 (8.3% gain share) — the only representative ticker where the regime indicator ranks among the top 3 features. Gold volatility exhibits well-documented regime sensitivity (flight-to-safety dynamics during high-VIX periods), and the constrained model correctly captures the regime-conditional structure.
- <span style="color: darkgreen;"><strong>✓</strong></span> MACRO__vixcls appears at rank 3 for SPY (5.3% gain share) and MACRO__t10y2y at rank 7 for TLT (3.6% gain share) — confirming that the exogenous macro conditioning variables contribute meaningfully to the constrained forecasts, with the yield-curve spread naturally more relevant for fixed-income assets.
- <span style="color: darkgreen;"><strong>✓</strong></span> No MOM__, VAL__, ML__, or SENT__ features appear anywhere in the importance table — the forbidden-prefix purity check passed, providing artifact-level proof that the DAG constraint held through model training.
</div>

**Next step:** CODE_BLOCK_I computes Shannon entropy and concentration statistics for constrained versus unconstrained feature-importance vectors to test Hypothesis H4 (interpretability via more balanced factor loadings).


***

In [21]:
# ============
# CODE_BLOCK_I
# ============

# ==============================================================================
# DEFINE HELPER TO COMPUTE ENTROPY AND CONCENTRATION STATISTICS FROM GAIN SHARES
# ==============================================================================

def compute_entropy_stats(fi_df: pd.DataFrame) -> Dict[str, float]:
    # ========================================================
    # RETURN NA-LIKE VALUES WHEN THE IMPORTANCE TABLE IS EMPTY
    # ========================================================

    if fi_df.empty:
        return {
            "entropy": float("nan"),
            "normalized_entropy": float("nan"),
            "non_zero_feature_count": 0,
            "top_1_gain_share": float("nan"),
            "top_5_gain_share": float("nan"),
        }

    # ===============================================
    # NORMALIZE GAIN VALUES INTO A PROBABILITY VECTOR
    # ===============================================

    p = fi_df["gain_share"].astype(float).values
    p = p[p > 0]

    # ================================================
    # COMPUTE SHANNON ENTROPY USING NATURAL LOGARITHMS
    # ================================================

    entropy = float(-(p * np.log(p)).sum())
    normalized_entropy = float(entropy / np.log(len(p))) if len(p) > 1 else 0.0

    # ===========================================================
    # RETURN ENTROPY PLUS SIMPLE CONCENTRATION SUMMARY STATISTICS
    # ===========================================================

    return {
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "non_zero_feature_count": int(len(p)),
        "top_1_gain_share": float(np.sort(p)[::-1][:1].sum()),
        "top_5_gain_share": float(np.sort(p)[::-1][:5].sum()),
    }

# ==================================================================
# FIT UNCONSTRAINED FINAL MODELS FOR THE SAME REPRESENTATIVE TICKERS
# ==================================================================

UNCONSTRAINED_IMPORTANCE_DFS: List[pd.DataFrame] = []

for ticker in REP_TICKERS_IMPORTANCE:
    # ==============================================================
    # REUSE NOTEBOOK 03 SAVED SPY IMPORTANCE ARTIFACT WHEN AVAILABLE
    # ==============================================================

    if ticker == "SPY" and NB03_BASELINE_IMPORTANCE_DF is not None:
        baseline_spy_fi_df = NB03_BASELINE_IMPORTANCE_DF.copy()
        baseline_spy_fi_df["ticker"] = "SPY"
        baseline_spy_fi_df["model_variant"] = "BASELINE_UNCONSTRAINED_ARTIFACT"
        baseline_spy_fi_df["dag_prefix"] = baseline_spy_fi_df["feature"].map(extract_feature_prefix)
        baseline_spy_fi_df["gain_share"] = baseline_spy_fi_df["gain"] / baseline_spy_fi_df["gain"].sum()
        baseline_spy_fi_df["rank_within_ticker"] = np.arange(1, len(baseline_spy_fi_df) + 1)
        UNCONSTRAINED_IMPORTANCE_DFS.append(
            baseline_spy_fi_df[["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"]]
        )
        continue

    # =============================================================
    # FIT A FRESH UNCONSTRAINED MODEL WHEN NO SAVED ARTIFACT EXISTS
    # =============================================================

    x_full = EFF_FEATURES_DF.copy()
    y_full = EFF_TARGET_DF[TARGET_COL_MAP[ticker]].copy()
    unconstrained_model = build_xgb_regressor(FROZEN_XGB_PARAMS)
    unconstrained_model.fit(x_full, y_full)
    unconstrained_gain_dict = unconstrained_model.get_booster().get_score(importance_type="gain")
    unconstrained_fi_df = gain_dict_to_dataframe(
        unconstrained_gain_dict,
        ticker=ticker,
        model_variant="BASELINE_UNCONSTRAINED_REFIT",
    )
    UNCONSTRAINED_IMPORTANCE_DFS.append(unconstrained_fi_df)

# ==================================================================
# CONCATENATE UNCONSTRAINED IMPORTANCE TABLES FOR ENTROPY COMPARISON
# ==================================================================

UNCONSTRAINED_FEATURE_IMPORTANCE_DF = pd.concat(UNCONSTRAINED_IMPORTANCE_DFS, ignore_index=True)

# ==============================================================================
# COMPUTE PER-TICKER ENTROPY FOR CONSTRAINED AND UNCONSTRAINED IMPORTANCE TABLES
# ==============================================================================

ENTROPY_ROWS: List[Dict[str, float]] = []

for model_variant, source_df in [
    ("CAUSAL_CONSTRAINED", CAUSAL_FEATURE_IMPORTANCE_DF),
    ("BASELINE_UNCONSTRAINED", UNCONSTRAINED_FEATURE_IMPORTANCE_DF),
]:
    # ==============================================
    # GROUP BY TICKER AND COMPUTE ENTROPY STATISTICS
    # ==============================================

    for ticker, ticker_fi_df in source_df.groupby("ticker", sort=True):
        entropy_stats = compute_entropy_stats(ticker_fi_df)
        ENTROPY_ROWS.append(
            {
                "ticker": ticker,
                "model_variant": model_variant,
                **entropy_stats,
            }
        )

# ======================================================
# BUILD WIDE ENTROPY COMPARISON TABLE WITH DELTA COLUMNS
# ======================================================

ENTROPY_LONG_DF = pd.DataFrame(ENTROPY_ROWS)

ENTROPY_WIDE_DF = (
    ENTROPY_LONG_DF.pivot(
        index="ticker",
        columns="model_variant",
        values=["entropy", "normalized_entropy", "non_zero_feature_count", "top_1_gain_share", "top_5_gain_share"],
    )
    .reset_index()
)

ENTROPY_WIDE_DF.columns = [
    "ticker" if col == ("ticker", "") else f"{col[0]}__{col[1]}"
    for col in ENTROPY_WIDE_DF.columns
]

ENTROPY_WIDE_DF["entropy_delta_causal_minus_unconstrained"] = (
    ENTROPY_WIDE_DF["entropy__CAUSAL_CONSTRAINED"] - ENTROPY_WIDE_DF["entropy__BASELINE_UNCONSTRAINED"]
)
ENTROPY_WIDE_DF["normalized_entropy_delta_causal_minus_unconstrained"] = (
    ENTROPY_WIDE_DF["normalized_entropy__CAUSAL_CONSTRAINED"] - ENTROPY_WIDE_DF["normalized_entropy__BASELINE_UNCONSTRAINED"]
)

# ===========================================================
# SAVE THE ENTROPY COMPARISON TABLE FOR NOTEBOOK 04 REPORTING
# ===========================================================

save_dataframe_csv(ENTROPY_WIDE_DF, NB04_ENTROPY_PATH)

# ===========================================================================
# PRINT THE ENTROPY OUTPUT PATH AND DISPLAY THE TICKER-LEVEL COMPARISON TABLE
# ===========================================================================

print("NB04_ENTROPY_PATH:", str(NB04_ENTROPY_PATH))
display(ENTROPY_WIDE_DF)

NB04_ENTROPY_PATH: /content/riskml-capstone/reports/tables/notebook04_entropy_comparison.csv


,ticker,entropy__BASELINE_UNCONSTRAINED,entropy__CAUSAL_CONSTRAINED,normalized_entropy__BASELINE_UNCONSTRAINED,normalized_entropy__CAUSAL_CONSTRAINED,non_zero_feature_count__BASELINE_UNCONSTRAINED,non_zero_feature_count__CAUSAL_CONSTRAINED,top_1_gain_share__BASELINE_UNCONSTRAINED,top_1_gain_share__CAUSAL_CONSTRAINED,top_5_gain_share__BASELINE_UNCONSTRAINED,top_5_gain_share__CAUSAL_CONSTRAINED,entropy_delta_causal_minus_unconstrained,normalized_entropy_delta_causal_minus_unconstrained
0,GLD,4.446337,3.826069,0.886205,0.888943,151.000000,74.000000,0.062777,0.092995,0.204204,0.295839,-0.620269,0.002738
1,SPY,4.037969,3.516566,0.804812,0.817034,151.000000,74.000000,0.092252,0.205927,0.329251,0.396003,-0.521403,0.012221
2,TLT,4.039234,3.602212,0.806132,0.836932,150.000000,74.000000,0.068715,0.133362,0.299400,0.403861,-0.437022,0.030800


### 📌 Observations and Insights for CODE_BLOCK_I

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Shannon entropy and normalized entropy computed for all three representative tickers under both constrained and unconstrained models. Entropy comparison table saved to artifact. Hypothesis H4 results: normalized entropy increases for all three tickers.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Defined `compute_entropy_stats()` to compute Shannon entropy $H = -\sum p_k \ln(p_k)$, normalized entropy $H / \ln(K)$, non-zero feature count, top-1 gain share, and top-5 gain share from a feature-importance table.
- For SPY, reused the Notebook 03 saved importance artifact (`BASELINE_UNCONSTRAINED_ARTIFACT`). For TLT and GLD, fit fresh unconstrained models on the full 151-feature effective window to generate comparable importance vectors.
- Computed per-ticker entropy for both `CAUSAL_CONSTRAINED` and `BASELINE_UNCONSTRAINED` model variants.
- Built a wide-format entropy comparison table with delta columns: `entropy_delta_causal_minus_unconstrained` and `normalized_entropy_delta_causal_minus_unconstrained`.
- Saved the entropy comparison to `notebook04_entropy_comparison.csv`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- **GLD**: Normalized entropy constrained = 0.889 vs unconstrained = 0.886 → delta = **+0.003**
- **SPY**: Normalized entropy constrained = 0.817 vs unconstrained = 0.805 → delta = **+0.012**
- **TLT**: Normalized entropy constrained = 0.837 vs unconstrained = 0.806 → delta = **+0.031** (largest increase)
- Raw entropy deltas are all negative (−0.62, −0.52, −0.44) because the constrained model uses fewer features (74 vs 151)
- Saved artifact: `notebook04_entropy_comparison.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> **Normalized entropy increases for all three tickers** (+0.003 for GLD, +0.012 for SPY, +0.031 for TLT). Normalized entropy adjusts for the number of features by dividing by $\ln(K)$, making the comparison fair despite the 74 vs 151 feature asymmetry. Higher normalized entropy means the constrained model distributes importance more evenly across the features available to the model — supporting Hypothesis H4.
- <span style="color: darkorange;"><strong>⚠</strong></span> Raw entropy decreases for all three tickers because Shannon entropy is mechanically sensitive to the number of features: $\ln(74) < \ln(151)$. The raw entropy decrease does not indicate worse interpretability — normalized entropy is the appropriate metric for cross-model comparison with different feature counts.
- <span style="color: darkgreen;"><strong>✓</strong></span> Top-1 gain share increases under constraints (e.g., SPY: 20.6% constrained vs 9.2% unconstrained), reflecting that a smaller feature pool necessarily concentrates more gain on the top feature. The top-5 gain share also increases. Despite this top-feature concentration, the remaining features are more evenly distributed, which is why normalized entropy still rises.
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> TLT shows the largest normalized entropy gain (+0.031), consistent with the constrained model eliminating 77 noise features (momentum, value, ML) that the unconstrained model distributed small but wasteful importance across. For fixed-income assets, the DAG constraint produces the most interpretable feature-importance profile.
</div>

**Next step:** CODE_BLOCK_J prepares the combined long-form prediction table (EWMA + XGBoost + Causal XGBoost for SPY), prefix-level gain share comparison, and top-feature comparison tables used by the final figure cells.


***

In [22]:
# ============
# CODE_BLOCK_J
# ============

# ================================================================
# SELECT A REPRESENTATIVE PLOT TICKER WITH SPY AS THE FIRST CHOICE
# ================================================================

PLOT_TICKER = "SPY" if "SPY" in TICKER_LIST else TICKER_LIST[0]

# ============================================================================
# BUILD A COMBINED LONG-FORM PREDICTION TABLE FOR CAUSAL XGBOOST AND BASELINES
# ============================================================================

PLOT_LONG_PARTS: List[pd.DataFrame] = [CAUSAL_PREDICTIONS_LONG_DF.copy()]

if NB03_BASELINE_PREDICTIONS_DF is not None:
    PLOT_LONG_PARTS.append(NB03_BASELINE_PREDICTIONS_DF.copy())

PLOT_LONG_DF = pd.concat(PLOT_LONG_PARTS, ignore_index=True)

# ============================================================================
# FILTER THE PLOT TABLE TO THE REPRESENTATIVE TICKER FOR FINAL FIGURE CREATION
# ============================================================================

PLOT_TICKER_DF = PLOT_LONG_DF[PLOT_LONG_DF["ticker"] == PLOT_TICKER].copy()
PLOT_TICKER_DF["date"] = pd.to_datetime(PLOT_TICKER_DF["date"])
PLOT_TICKER_DF = PLOT_TICKER_DF.sort_values(["date", "model"]).reset_index(drop=True)

# =========================================================================
# COMPUTE PREFIX-LEVEL GAIN SHARES FOR CONSTRAINED AND UNCONSTRAINED MODELS
# =========================================================================

PREFIX_GAIN_SHARE_DF = (
    pd.concat([CAUSAL_FEATURE_IMPORTANCE_DF, UNCONSTRAINED_FEATURE_IMPORTANCE_DF], ignore_index=True)
    .assign(
        model_family=lambda df: df["model_variant"].map(
            lambda value: "CAUSAL_CONSTRAINED" if "CAUSAL" in value else "BASELINE_UNCONSTRAINED"
        )
    )
    .groupby(["ticker", "model_family", "dag_prefix"], dropna=False)["gain_share"]
    .sum()
    .reset_index()
    .sort_values(["ticker", "model_family", "gain_share"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# =================================================================
# SAVE PREFIX GAIN SHARE TABLE FOR DOWNSTREAM PLOTS AND REPORT TEXT
# =================================================================

save_dataframe_csv(PREFIX_GAIN_SHARE_DF, NB04_PREFIX_GAIN_PATH)

# =====================================================================
# BUILD TOP-FEATURE COMPARISON TABLE FOR THE REPRESENTATIVE PLOT TICKER
# =====================================================================

TOP_FEATURE_COMPARISON_DF = (
    pd.concat([CAUSAL_FEATURE_IMPORTANCE_DF, UNCONSTRAINED_FEATURE_IMPORTANCE_DF], ignore_index=True)
    .assign(
        model_family=lambda df: df["model_variant"].map(
            lambda value: "CAUSAL_CONSTRAINED" if "CAUSAL" in value else "BASELINE_UNCONSTRAINED"
        )
    )
    .query("ticker == @PLOT_TICKER")
    .sort_values(["model_family", "gain"], ascending=[True, False])
    .groupby("model_family", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

# ==============================================================================
# SAVE THE TOP-FEATURE COMPARISON TABLE FOR WORD INSERTION AND NARRATIVE SUPPORT
# ==============================================================================

save_dataframe_csv(TOP_FEATURE_COMPARISON_DF, NB04_TOP_FEATURE_COMPARISON_PATH)

# ===============================================================================
# PRINT PLOT-TICKER OUTPUT PATHS AND DISPLAY TABLES USED BY THE FINAL FIGURE CELL
# ===============================================================================

print("PLOT_TICKER:", PLOT_TICKER)
print("NB04_PREFIX_GAIN_PATH:", str(NB04_PREFIX_GAIN_PATH))
print("NB04_TOP_FEATURE_COMPARISON_PATH:", str(NB04_TOP_FEATURE_COMPARISON_PATH))
display(PLOT_TICKER_DF.head(20))
display(PREFIX_GAIN_SHARE_DF.head(20))
display(TOP_FEATURE_COMPARISON_DF)

PLOT_TICKER: SPY
NB04_PREFIX_GAIN_PATH: /content/riskml-capstone/reports/tables/notebook04_prefix_gain_share.csv
NB04_TOP_FEATURE_COMPARISON_PATH: /content/riskml-capstone/reports/tables/notebook04_top_feature_comparison.csv


,date,ticker,model,y_true,y_pred
0,2023-12-28,SPY,CAUSAL_XGBOOST,0.095915,0.124487
1,2023-12-28,SPY,EWMA,0.095915,0.111554
2,2023-12-28,SPY,XGBOOST,0.095915,0.146426
3,2023-12-29,SPY,CAUSAL_XGBOOST,0.094912,0.114656
4,2023-12-29,SPY,EWMA,0.094912,0.110570
5,2023-12-29,SPY,XGBOOST,0.094912,0.137593
6,2024-01-02,SPY,CAUSAL_XGBOOST,0.112099,0.110453
7,2024-01-02,SPY,EWMA,0.112099,0.110688
8,2024-01-02,SPY,XGBOOST,0.112099,0.129859
9,2024-01-03,SPY,CAUSAL_XGBOOST,0.114090,0.108563


,ticker,model_family,dag_prefix,gain_share
0,GLD,BASELINE_UNCONSTRAINED,VOL,0.528396
1,GLD,BASELINE_UNCONSTRAINED,MOM,0.302759
2,GLD,BASELINE_UNCONSTRAINED,VAL,0.132472
3,GLD,BASELINE_UNCONSTRAINED,MACRO,0.030439
4,GLD,BASELINE_UNCONSTRAINED,REGIME,0.004711
5,GLD,BASELINE_UNCONSTRAINED,ML,0.001223
6,GLD,CAUSAL_CONSTRAINED,VOL,0.875861
7,GLD,CAUSAL_CONSTRAINED,REGIME,0.083404
8,GLD,CAUSAL_CONSTRAINED,MACRO,0.040735
9,SPY,BASELINE_UNCONSTRAINED,VOL,0.469409


,ticker,model_variant,feature,dag_prefix,gain,gain_share,rank_within_ticker,model_path,model_family
0,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,VOL__XLK__ewma_vol__span21,VOL,0.550756,0.092252,1,NaN,BASELINE_UNCONSTRAINED
1,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,MOM__XLF__cum_ret__10d,MOM,0.408763,0.068468,2,NaN,BASELINE_UNCONSTRAINED
2,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,VOL__QQQ__ewma_vol__span21,VOL,0.401745,0.067292,3,NaN,BASELINE_UNCONSTRAINED
3,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,VOL__SPY__ewma_vol__span21,VOL,0.357434,0.059870,4,NaN,BASELINE_UNCONSTRAINED
4,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,MACRO__vixcls,MACRO,0.246977,0.041369,5,NaN,BASELINE_UNCONSTRAINED
5,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,REGIME__vix_high,REGIME,0.223900,0.037503,6,NaN,BASELINE_UNCONSTRAINED
6,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,VAL__TLT__hml_beta__63d,VAL,0.196913,0.032983,7,NaN,BASELINE_UNCONSTRAINED
7,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,MOM__SPY__cum_ret__21d,MOM,0.194209,0.032530,8,NaN,BASELINE_UNCONSTRAINED
8,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,MOM__SPY__cum_ret__10d,MOM,0.179289,0.030031,9,NaN,BASELINE_UNCONSTRAINED
9,SPY,BASELINE_UNCONSTRAINED_ARTIFACT,VAL__ff_rf,VAL,0.148781,0.024921,10,NaN,BASELINE_UNCONSTRAINED


### 📌 Observations and Insights for CODE_BLOCK_J

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Combined prediction table, prefix gain share, and top-feature comparison tables built and saved for SPY. All three model variants (EWMA, XGBOOST, CAUSAL_XGBOOST) present in the prediction overlay data.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Concatenated the Notebook 04 causal predictions with the Notebook 03 baseline predictions into a single long-form table, then filtered to the representative ticker (SPY) for figure preparation.
- Computed prefix-level gain shares by aggregating feature-level gain shares to DAG-prefix level for both constrained and unconstrained models across all three representative tickers.
- Built a top-10 feature comparison table for SPY showing the highest-gain features side-by-side for BASELINE_UNCONSTRAINED and CAUSAL_CONSTRAINED model families.
- Saved `notebook04_prefix_gain_share.csv` and `notebook04_top_feature_comparison.csv` to `reports/tables/`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `PLOT_TICKER`: SPY
- **SPY prefix gain shares (constrained)**: VOL = 89.1%, MACRO = 8.7%, REGIME = 2.2%
- **SPY prefix gain shares (unconstrained)**: VOL = 46.9%, MOM = 33.9%, VAL = 10.6%, MACRO = 4.7%, REGIME = 3.8%, ML = 0.1%
- **Unconstrained SPY rank-2 feature**: `MOM__XLF__cum_ret__10d` (6.8% gain share) — a momentum cross-node shortcut
- **Constrained SPY rank-2 feature**: `VOL__XLV__rvol__10d` (5.5% gain share) — a volatility-family feature
- Saved artifacts: `notebook04_prefix_gain_share.csv`, `notebook04_top_feature_comparison.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The prefix gain share comparison reveals the DAG constraint's primary effect: the unconstrained SPY model allocates 33.9% of gain to MOM__ features and 10.6% to VAL__ features — cross-node information paths that the DAG forbids at the Risk stage. The constrained model redirects that 44.5% of gain to VOL__ features (from 46.9% → 89.1%), producing a more focused and interpretable risk model.
- <span style="color: darkgreen;"><strong>✓</strong></span> The top-feature comparison for SPY confirms the shift: the unconstrained model's rank-2 feature is `MOM__XLF__cum_ret__10d` (a momentum feature), while the constrained model's rank-2 feature is `VOL__XLV__rvol__10d` (a volatility feature). The DAG constraint successfully eliminated the momentum-to-risk shortcut identified in the Notebook 03 baseline analysis.
- <span style="color: darkgreen;"><strong>✓</strong></span> The combined prediction preview shows all three models producing forecasts for the same SPY test dates, confirming that the CODE_BLOCK_N forecast overlay figure will display a clean three-model comparison.
</div>

**Next step:** CODE_BLOCK_K builds the pivot tables, delta tables, and aggregate summary used by the final figure and narrative cells.


***

In [23]:
# ============
# CODE_BLOCK_K
# ============

# =============================================================================
# BUILD A PIVOT TABLE OF MODEL PREDICTIONS FOR THE REPRESENTATIVE FORECAST PLOT
# =============================================================================

PLOT_PIVOT_DF = PLOT_TICKER_DF.pivot_table(index="date", columns="model", values="y_pred", aggfunc="first")

# ==============================================================================
# EXTRACT THE REALIZED TARGET SERIES ONCE PER DATE FOR THE REPRESENTATIVE TICKER
# ==============================================================================

REALIZED_PLOT_SERIES = (
    PLOT_TICKER_DF.drop_duplicates(subset=["date"])[["date", "y_true"]]
    .set_index("date")["y_true"]
    .sort_index()
)

# =============================================================
# CREATE RMSE AND MAE DELTA TABLES USED BY THE FINAL BAR CHARTS
# =============================================================

RMSE_DELTA_PLOT_DF = (
    CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"]
    .loc[:, ["ticker", "causal_minus_xgboost_rmse", "causal_minus_ewma_rmse"]]
    .sort_values("causal_minus_xgboost_rmse")
    .reset_index(drop=True)
)

MAE_DELTA_PLOT_DF = (
    CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"]
    .loc[:, ["ticker", "causal_minus_xgboost_mae", "causal_minus_ewma_mae"]]
    .sort_values("causal_minus_xgboost_mae")
    .reset_index(drop=True)
)

# =======================================================================
# BUILD A COMPACT AGGREGATE SUMMARY TABLE FOR NOTEBOOK 04 NARRATIVE CELLS
# =======================================================================

AGGREGATE_SUMMARY_DF = CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] == "AGGREGATE_EQUAL_WEIGHT"].copy()

# ======================================================
# PRINT SUMMARY STATISTICS AND DISPLAY PLOT-READY TABLES
# ======================================================

print("PLOT_PIVOT_COLUMNS:", list(PLOT_PIVOT_DF.columns))
print("REPRESENTATIVE_REALIZED_OBSERVATIONS:", len(REALIZED_PLOT_SERIES))
display(RMSE_DELTA_PLOT_DF)
display(MAE_DELTA_PLOT_DF)
display(AGGREGATE_SUMMARY_DF)

PLOT_PIVOT_COLUMNS: ['CAUSAL_XGBOOST', 'EWMA', 'XGBOOST']
REPRESENTATIVE_REALIZED_OBSERVATIONS: 500


,ticker,causal_minus_xgboost_rmse,causal_minus_ewma_rmse
0,GLD,0.000372,-0.004467
1,XLV,0.003267,0.006016
2,DBC,0.004702,-0.003215
3,QQQ,0.005271,-0.007809
4,LQD,0.008803,0.020816
5,SPY,0.008986,-0.001127
6,XLK,0.010636,-0.001829
7,IWM,0.010927,0.008706
8,HYG,0.011039,0.017323
9,TLT,0.011748,0.022361


,ticker,causal_minus_xgboost_mae,causal_minus_ewma_mae
0,IWM,-0.004430,-0.008646
1,GLD,-0.001526,-0.005455
2,QQQ,-0.001506,-0.008155
3,XLV,-0.001336,0.005396
4,XLK,-0.000513,-0.008392
5,EEM,-0.000496,0.000885
6,XLF,0.000231,0.008207
7,LQD,0.001777,0.006925
8,EFA,0.002377,0.004087
9,SPY,0.002699,-0.003195


,model,ticker,causal_rmse,causal_mae,causal_n_obs,feature_count,xgboost_rmse,xgboost_mae,xgboost_n_obs,causal_minus_xgboost_rmse,causal_minus_xgboost_mae,rmse_improves_vs_xgboost,mae_improves_vs_xgboost,ewma_rmse,ewma_mae,ewma_n_obs,causal_minus_ewma_rmse,causal_minus_ewma_mae,rmse_improves_vs_ewma,mae_improves_vs_ewma
14,CAUSAL_XGBOOST,AGGREGATE_EQUAL_WEIGHT,0.081112,0.050080,500,74,0.069814,0.049100,500,0.011298,0.000980,False,False,0.070824,0.048309,500,0.010288,0.001772,False,False


### 📌 Observations and Insights for CODE_BLOCK_K

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — Pivot table contains all three model columns (CAUSAL_XGBOOST, EWMA, XGBOOST). Realized series has 500 observations. RMSE and MAE delta tables sorted and ready for bar chart plotting.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Pivoted the long-form SPY prediction table into a date-indexed wide-format table with one column per model (CAUSAL_XGBOOST, EWMA, XGBOOST) for clean time-series plotting.
- Extracted the realized forward-volatility series (one value per date, deduplicated) as the ground-truth reference line.
- Built RMSE and MAE delta tables sorted by causal-minus-XGBoost delta for the 14 non-aggregate tickers — these tables drive the bar charts in CODE_BLOCK_N.
- Isolated the AGGREGATE_EQUAL_WEIGHT row into a compact summary table for narrative cells.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `PLOT_PIVOT_COLUMNS`: ['CAUSAL_XGBOOST', 'EWMA', 'XGBOOST']
- `REPRESENTATIVE_REALIZED_OBSERVATIONS`: 500
- RMSE delta range versus XGBoost: GLD (+0.0004, nearly break-even) to XLE (+0.0286, largest degradation)
- MAE delta range versus XGBoost: IWM (−0.0044, best improvement) to DBC (+0.0045, worst degradation)
- Aggregate: Causal RMSE 0.0811, XGBoost RMSE 0.0698, EWMA RMSE 0.0708
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> GLD shows a near-zero RMSE delta versus XGBoost (+0.0004), indicating that DAG constraints impose essentially no accuracy cost for commodity-gold volatility forecasting — the unconstrained model's momentum and value features added negligible signal for GLD.
- <span style="color: darkorange;"><strong>⚠</strong></span> XLE (+0.0286) and XLF (+0.0230) show the largest RMSE degradation, suggesting that the unconstrained model's access to sector-specific momentum features was particularly informative for energy and financial sector volatility. The report narrative should acknowledge these sector-specific costs.
- <span style="color: darkgreen;"><strong>✓</strong></span> On MAE, 6 of 14 tickers show improvement (negative delta) versus XGBoost: IWM, GLD, QQQ, XLV, XLK, EEM — indicating that the constrained model produces more accurate median-level forecasts for a substantial subset of the ETF universe even while aggregate RMSE increases.
</div>

**Next step:** CODE_BLOCK_L maps tickers to asset classes and computes asset-class-level RMSE/MAE delta summaries for structured reporting.


***

In [24]:
# ============
# CODE_BLOCK_L
# ============

# ==============================================================
# DEFINE SIMPLE ASSET-CLASS LABELS FOR THE CAPSTONE ETF UNIVERSE
# ==============================================================

ASSET_CLASS_MAP = {
    "SPY": "US_EQUITY",
    "QQQ": "US_EQUITY",
    "IWM": "US_EQUITY",
    "EFA": "INTERNATIONAL_EQUITY",
    "EEM": "INTERNATIONAL_EQUITY",
    "XLK": "SECTOR_EQUITY",
    "XLF": "SECTOR_EQUITY",
    "XLE": "SECTOR_EQUITY",
    "XLV": "SECTOR_EQUITY",
    "TLT": "FIXED_INCOME",
    "LQD": "FIXED_INCOME",
    "HYG": "FIXED_INCOME",
    "GLD": "COMMODITY",
    "DBC": "COMMODITY",
}

# =========================================================================
# MERGE ASSET-CLASS LABELS INTO THE CAUSAL VERSUS BASELINE COMPARISON TABLE
# =========================================================================

CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF = CAUSAL_VS_BASELINE_DF.copy()
CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF["asset_class"] = CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF["ticker"].map(ASSET_CLASS_MAP).fillna("UNMAPPED")

# ==========================================================================
# BUILD ASSET-CLASS SUMMARY STATISTICS FOR RMSE AND MAE DELTA INTERPRETATION
# ==========================================================================

ASSET_CLASS_SUMMARY_DF = (
    CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF.query('ticker != "AGGREGATE_EQUAL_WEIGHT"')
    .groupby("asset_class", dropna=False)
    .agg(
        ticker_count=("ticker", "count"),
        mean_causal_minus_xgboost_rmse=("causal_minus_xgboost_rmse", "mean"),
        mean_causal_minus_ewma_rmse=("causal_minus_ewma_rmse", "mean"),
        mean_causal_minus_xgboost_mae=("causal_minus_xgboost_mae", "mean"),
        mean_causal_minus_ewma_mae=("causal_minus_ewma_mae", "mean"),
        rmse_wins_vs_xgboost=("rmse_improves_vs_xgboost", "sum"),
        rmse_wins_vs_ewma=("rmse_improves_vs_ewma", "sum"),
    )
    .reset_index()
    .sort_values("mean_causal_minus_xgboost_rmse")
    .reset_index(drop=True)
)

# ==============================================================
# SAVE THE ASSET-CLASS SUMMARY TABLE FOR REPORT SUBSECTION REUSE
# ==============================================================

save_dataframe_csv(ASSET_CLASS_SUMMARY_DF, NB04_ASSET_CLASS_PATH)

# ===============================================================
# PRINT THE ASSET-CLASS OUTPUT PATH AND DISPLAY THE SUMMARY TABLE
# ===============================================================

print("NB04_ASSET_CLASS_PATH:", str(NB04_ASSET_CLASS_PATH))
display(ASSET_CLASS_SUMMARY_DF)

NB04_ASSET_CLASS_PATH: /content/riskml-capstone/reports/tables/notebook04_asset_class_comparison.csv


,asset_class,ticker_count,mean_causal_minus_xgboost_rmse,mean_causal_minus_ewma_rmse,mean_causal_minus_xgboost_mae,mean_causal_minus_ewma_mae,rmse_wins_vs_xgboost,rmse_wins_vs_ewma
0,COMMODITY,2,0.002537,-0.003841,0.001507,-0.002858,0,2
1,US_EQUITY,3,0.008395,-0.000077,-0.001079,-0.006665,0,2
2,FIXED_INCOME,3,0.010530,0.020167,0.003137,0.009094,0,0
3,INTERNATIONAL_EQUITY,2,0.015378,0.011587,0.000941,0.002486,0,0
4,SECTOR_EQUITY,4,0.016391,0.017068,0.000664,0.004566,0,1


### 📌 Observations and Insights for CODE_BLOCK_L

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — All 14 tickers mapped to asset classes. Asset-class summary table saved. XLV mapped to UNMAPPED (minor labeling issue, does not affect metrics).
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Defined asset-class labels for the 14-ETF universe: US_EQUITY (SPY, QQQ, IWM), INTERNATIONAL_EQUITY (EFA, EEM), SECTOR_EQUITY (XLF, XLK, XLE), FIXED_INCOME (TLT, LQD, HYG), COMMODITY (GLD, DBC). XLV mapped to UNMAPPED because the `ASSET_CLASS_MAP` includes VNQ (Real Estate, absent from the universe) instead of XLV (Health Care).
- Aggregated RMSE and MAE deltas by asset class, computing mean deltas and win counts for structured report subsection writing.
- Saved `notebook04_asset_class_comparison.csv` to `reports/tables/`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- **COMMODITY** (GLD, DBC): Mean RMSE delta vs XGBoost = +0.003 (smallest degradation); RMSE wins vs EWMA = 2
- **US_EQUITY** (SPY, QQQ, IWM): Mean RMSE delta vs XGBoost = +0.008; RMSE wins vs EWMA = 2
- **FIXED_INCOME** (TLT, LQD, HYG): Mean RMSE delta vs XGBoost = +0.011; RMSE wins vs EWMA = 0
- **INTERNATIONAL_EQUITY** (EFA, EEM): Mean RMSE delta vs XGBoost = +0.015; RMSE wins vs EWMA = 0
- **SECTOR_EQUITY** (XLF, XLK, XLE): Mean RMSE delta vs XGBoost = +0.021 (largest degradation); RMSE wins vs EWMA = 1
- Saved artifact: `notebook04_asset_class_comparison.csv`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> Commodities show the smallest accuracy cost under DAG constraints (+0.003 mean RMSE delta), consistent with the empirical structure of commodity volatility — gold and broad-commodity ETFs are driven primarily by their own volatility persistence and VIX-regime dynamics, exactly the features the constrained model retains.
- <span style="color: darkorange;"><strong>⚠</strong></span> Sector equity shows the largest accuracy cost (+0.021), likely because sector ETFs (XLF, XLK, XLE) have strong momentum and cross-sector factor exposure that the unconstrained model exploits but the DAG forbids at the Risk node.
- <span style="color: darkorange;"><strong>⚠</strong></span> The UNMAPPED ticker (XLV) should be classified as SECTOR_EQUITY in the `ASSET_CLASS_MAP` for a future cleanup pass — XLV (Health Care Select Sector SPDR) belongs in the same category as XLF, XLK, and XLE. This labeling issue does not affect accuracy; the ticker-level metrics are correct regardless of the label.
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The asset-class decomposition provides a structured narrative for the capstone report: DAG constraints impose minimal cost on assets driven by volatility dynamics (commodities, broad equity) but higher cost on sector assets where cross-node factor exposure is empirically important.
</div>

**Next step:** CODE_BLOCK_M defines all six figure output paths and assembles the preliminary artifact registry with file-existence checks before figure creation.


***

In [25]:
# ============
# CODE_BLOCK_M
# ============

# ========================================================================
# DEFINE THE FINAL NOTEBOOK 04 FIGURE PATHS USED BY THE LAST ARTIFACT CELL
# ========================================================================

NB04_DAG_FIG_PATH = FIGURES_DIR / "notebook04_dag_figure.png"
NB04_FORECAST_FIG_PATH = FIGURES_DIR / f"notebook04_forecast_vs_realized_{PLOT_TICKER}.png"
NB04_RMSE_DELTA_FIG_PATH = FIGURES_DIR / "notebook04_causal_vs_baseline_rmse_delta.png"
NB04_ENTROPY_FIG_PATH = FIGURES_DIR / "notebook04_entropy_comparison.png"
NB04_CAUSAL_IMPORTANCE_FIG_PATH = FIGURES_DIR / f"notebook04_causal_feature_importance_{PLOT_TICKER}.png"
NB04_PREFIX_GAIN_FIG_PATH = FIGURES_DIR / f"notebook04_prefix_gain_share_{PLOT_TICKER}.png"

# ==========================================================================
# ASSEMBLE A PRELIMINARY ARTIFACT REGISTRY TABLE FOR FINAL MANIFEST CREATION
# ==========================================================================

ARTIFACT_REGISTRY_DF = pd.DataFrame(
    [
        {"artifact_type": "TABLE", "artifact_path": str(NB04_CAUSAL_METRICS_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_CAUSAL_PREDICTIONS_TABLE_PATH)},
        {"artifact_type": "DATA", "artifact_path": str(NB04_CAUSAL_PREDICTIONS_DATA_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_GATING_SUMMARY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_EFFECTIVE_WINDOW_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_SPLIT_SUMMARY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_COMPARISON_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_FEATURE_IMPORTANCE_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_ENTROPY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_PREFIX_GAIN_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_TOP_FEATURE_COMPARISON_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_ASSET_CLASS_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_DAG_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_FORECAST_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_RMSE_DELTA_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_ENTROPY_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_CAUSAL_IMPORTANCE_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_PREFIX_GAIN_FIG_PATH)},
    ]
)

# ==========================================================================
# ADD CURRENT FILE-EXISTS FLAGS SO THE FINAL CELL CAN VERIFY FIGURE CREATION
# ==========================================================================

ARTIFACT_REGISTRY_DF["exists_now"] = ARTIFACT_REGISTRY_DF["artifact_path"].map(lambda path: Path(path).exists())

# =======================================================
# PRINT PRELIMINARY REGISTRY STATUS AND DISPLAY THE TABLE
# =======================================================

print("PRELIMINARY_ARTIFACT_COUNT:", len(ARTIFACT_REGISTRY_DF))
display(ARTIFACT_REGISTRY_DF)

PRELIMINARY_ARTIFACT_COUNT: 18


,artifact_type,artifact_path,exists_now
0,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
1,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
2,DATA,/content/riskml-capstone/data/processed/notebo...,True
3,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
4,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
5,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
6,TABLE,/content/riskml-capstone/reports/tables/causal...,True
7,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
8,TABLE,/content/riskml-capstone/reports/tables/notebo...,True
9,TABLE,/content/riskml-capstone/reports/tables/notebo...,True


### 📌 Observations and Insights for CODE_BLOCK_M

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — 18 total artifacts registered. All 12 TABLE and DATA artifacts exist (True). All 6 FIGURE artifacts correctly show False (not yet created) — figures are generated in CODE_BLOCK_N.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Defined the six figure output paths: DAG diagram, forecast overlay (SPY), RMSE delta bar chart, entropy comparison, causal feature importance (SPY), and prefix gain share (SPY).
- Assembled an 18-row artifact registry DataFrame listing every Notebook 04 output (12 tables/data + 6 figures) with artifact type and absolute file path.
- Added `exists_now` boolean flags — all 12 pre-existing artifacts show True; all 6 pending figures show False, confirming that CODE_BLOCK_N is the sole figure-creation cell.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- `PRELIMINARY_ARTIFACT_COUNT`: 18 (12 True + 6 False)
- Figure paths defined for: `notebook04_dag_figure.png`, `notebook04_forecast_vs_realized_SPY.png`, `notebook04_causal_vs_baseline_rmse_delta.png`, `notebook04_entropy_comparison.png`, `notebook04_causal_feature_importance_SPY.png`, `notebook04_prefix_gain_share_SPY.png`
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> The 12/12 True status for table artifacts confirms that CODE_BLOCKs A through L executed without silent failures — every expected CSV was written to disk.
- <span style="color: darkgreen;"><strong>✓</strong></span> The 6/6 False status for figure artifacts is the expected state before CODE_BLOCK_N runs. After CODE_BLOCK_N executes, the final artifact manifest will confirm 18/18 True.
- <span style="color: darkgreen;"><strong>✓</strong></span> Centralizing the artifact registry in a single cell provides a single-table audit trail for the capstone report — the final manifest from CODE_BLOCK_N will include file sizes for completeness verification.
</div>

**Next step:** CODE_BLOCK_N generates all six Notebook 04 figures (DAG diagram, forecast overlay, RMSE delta bars, entropy bars, feature importance bars, prefix gain share bars) and produces the final artifact manifest confirming 18/18 artifacts exist.


***

In [26]:
# ============
# CODE_BLOCK_N
# ============

# ==========================================================================
# CREATE THE MANUAL DAG FIGURE FOR REPORT INSERTION AND PIPELINE EXPLANATION
# ==========================================================================

fig, ax = plt.subplots(figsize=(12, 4))
ax.set_title("MANUAL DAG FOR NOTEBOOK 04 RISK-STAGE FEATURE GATING")
ax.axis("off")

dag_positions = {
    "SENTIMENT": (0.08, 0.55),
    "MOMENTUM": (0.28, 0.55),
    "RETURNS": (0.48, 0.55),
    "VALUE": (0.28, 0.20),
    "VOLATILITY": (0.48, 0.20),
    "RISK": (0.68, 0.20),
    "ALLOCATION": (0.88, 0.20),
    "MACRO": (0.68, 0.55),
    "REGIME": (0.88, 0.55),
}

for node_name, (x_coord, y_coord) in dag_positions.items():
    # ================================================
    # DRAW A ROUNDED NODE BOX FOR THE CURRENT DAG NODE
    # ================================================

    node_box = FancyBboxPatch(
        (x_coord - 0.07, y_coord - 0.05),
        0.14,
        0.10,
        boxstyle="round,pad=0.02",
        transform=ax.transAxes,
    )
    ax.add_patch(node_box)
    ax.text(x_coord, y_coord, node_name, ha="center", va="center", transform=ax.transAxes)

dag_edges_for_plot = [
    ("SENTIMENT", "MOMENTUM"),
    ("MOMENTUM", "RETURNS"),
    ("VALUE", "RETURNS"),
    ("VOLATILITY", "RISK"),
    ("RISK", "ALLOCATION"),
    ("MACRO", "RISK"),
    ("REGIME", "RISK"),
]

for source_node, target_node in dag_edges_for_plot:
    # ===========================================================
    # DRAW A DIRECTED ARROW FOR THE CURRENT INFORMATION-FLOW EDGE
    # ===========================================================

    x0, y0 = dag_positions[source_node]
    x1, y1 = dag_positions[target_node]
    ax.annotate(
        "",
        xy=(x1 - 0.08, y1),
        xytext=(x0 + 0.08, y0),
        xycoords=ax.transAxes,
        textcoords=ax.transAxes,
        arrowprops={"arrowstyle": "->", "lw": 1.5},
    )

plt.tight_layout()
save_figure_png(fig, NB04_DAG_FIG_PATH, dpi=200)
plt.close(fig)

# ==================================================================
# CREATE FORECAST VERSUS REALIZED PLOT FOR THE REPRESENTATIVE TICKER
# ==================================================================

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(REALIZED_PLOT_SERIES.index, REALIZED_PLOT_SERIES.values, label="REALIZED_FWD_VOL_20D")

for model_name in ["EWMA", "XGBOOST", "CAUSAL_XGBOOST"]:
    # ======================================================
    # PLOT EACH AVAILABLE MODEL SERIES ON THE SAME DATE AXIS
    # ======================================================

    if model_name in PLOT_PIVOT_DF.columns:
        ax.plot(PLOT_PIVOT_DF.index, PLOT_PIVOT_DF[model_name].values, label=model_name)

ax.set_title(f"FORECAST VS REALIZED VOLATILITY FOR {PLOT_TICKER}")
ax.set_xlabel("DATE")
ax.set_ylabel("ANNUALIZED VOLATILITY")
ax.legend()
plt.tight_layout()
save_figure_png(fig, NB04_FORECAST_FIG_PATH, dpi=200)
plt.close(fig)

# ===============================================================
# CREATE RMSE DELTA BAR CHART RELATIVE TO THE TWO BASELINE MODELS
# ===============================================================

fig, ax = plt.subplots(figsize=(14, 5))
rmse_plot_df = RMSE_DELTA_PLOT_DF.set_index("ticker")
rmse_plot_df.plot(kind="bar", ax=ax)
ax.axhline(0.0, linewidth=1.0)
ax.set_title("CAUSAL MODEL RMSE DELTA VERSUS XGBOOST AND EWMA")
ax.set_xlabel("TICKER")
ax.set_ylabel("DELTA RMSE")
plt.xticks(rotation=45)
plt.tight_layout()
save_figure_png(fig, NB04_RMSE_DELTA_FIG_PATH, dpi=200)
plt.close(fig)

# ==================================================================
# CREATE ENTROPY COMPARISON BAR CHART FOR THE REPRESENTATIVE TICKERS
# ==================================================================

fig, ax = plt.subplots(figsize=(12, 5))
entropy_plot_df = ENTROPY_WIDE_DF.set_index("ticker")[
    [
        "normalized_entropy__CAUSAL_CONSTRAINED",
        "normalized_entropy__BASELINE_UNCONSTRAINED",
    ]
]
entropy_plot_df.plot(kind="bar", ax=ax)
ax.set_title("NORMALIZED FEATURE-IMPORTANCE ENTROPY COMPARISON")
ax.set_xlabel("TICKER")
ax.set_ylabel("NORMALIZED ENTROPY")
plt.xticks(rotation=0)
plt.tight_layout()
save_figure_png(fig, NB04_ENTROPY_FIG_PATH, dpi=200)
plt.close(fig)

# ======================================================================
# CREATE TOP-CAUSAL-FEATURE BAR CHART FOR THE REPRESENTATIVE PLOT TICKER
# ======================================================================

fig, ax = plt.subplots(figsize=(12, 6))
causal_top_plot_df = (
    CAUSAL_FEATURE_IMPORTANCE_DF.query("ticker == @PLOT_TICKER")
    .sort_values("gain", ascending=True)
    .tail(15)
)
ax.barh(causal_top_plot_df["feature"], causal_top_plot_df["gain"])
ax.set_title(f"TOP CAUSAL FEATURE IMPORTANCE FOR {PLOT_TICKER}")
ax.set_xlabel("GAIN")
ax.set_ylabel("FEATURE")
plt.tight_layout()
save_figure_png(fig, NB04_CAUSAL_IMPORTANCE_FIG_PATH, dpi=200)
plt.close(fig)

# =====================================================================
# CREATE PREFIX GAIN SHARE BAR CHART FOR THE REPRESENTATIVE PLOT TICKER
# =====================================================================

fig, ax = plt.subplots(figsize=(10, 5))
prefix_plot_df = (
    PREFIX_GAIN_SHARE_DF.query("ticker == @PLOT_TICKER")
    .pivot(index="dag_prefix", columns="model_family", values="gain_share")
    .fillna(0.0)
    .sort_index()
)
prefix_plot_df.plot(kind="bar", ax=ax)
ax.set_title(f"PREFIX GAIN SHARE COMPARISON FOR {PLOT_TICKER}")
ax.set_xlabel("DAG PREFIX")
ax.set_ylabel("TOTAL GAIN SHARE")
plt.xticks(rotation=45)
plt.tight_layout()
save_figure_png(fig, NB04_PREFIX_GAIN_FIG_PATH, dpi=200)
plt.close(fig)

# =========================================================================
# BUILD THE FINAL ARTIFACT MANIFEST AFTER ALL NOTEBOOK 04 FIGURES ARE SAVED
# =========================================================================

NOTEBOOK04_ARTIFACT_MANIFEST_DF = ARTIFACT_REGISTRY_DF.copy()
NOTEBOOK04_ARTIFACT_MANIFEST_DF["exists_now"] = NOTEBOOK04_ARTIFACT_MANIFEST_DF["artifact_path"].map(lambda path: Path(path).exists())
NOTEBOOK04_ARTIFACT_MANIFEST_DF["file_size_bytes"] = NOTEBOOK04_ARTIFACT_MANIFEST_DF["artifact_path"].map(
    lambda path: Path(path).stat().st_size if Path(path).exists() else np.nan
)
save_dataframe_csv(NOTEBOOK04_ARTIFACT_MANIFEST_DF, NB04_ARTIFACT_MANIFEST_PATH)

# ====================================================================
# PRINT FINAL FIGURE PATHS AND DISPLAY THE COMPLETED ARTIFACT MANIFEST
# ====================================================================

print("NB04_DAG_FIG_PATH:", str(NB04_DAG_FIG_PATH))
print("NB04_FORECAST_FIG_PATH:", str(NB04_FORECAST_FIG_PATH))
print("NB04_RMSE_DELTA_FIG_PATH:", str(NB04_RMSE_DELTA_FIG_PATH))
print("NB04_ENTROPY_FIG_PATH:", str(NB04_ENTROPY_FIG_PATH))
print("NB04_CAUSAL_IMPORTANCE_FIG_PATH:", str(NB04_CAUSAL_IMPORTANCE_FIG_PATH))
print("NB04_PREFIX_GAIN_FIG_PATH:", str(NB04_PREFIX_GAIN_FIG_PATH))
print("NB04_ARTIFACT_MANIFEST_PATH:", str(NB04_ARTIFACT_MANIFEST_PATH))
display(NOTEBOOK04_ARTIFACT_MANIFEST_DF)

NB04_DAG_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_dag_figure.png
NB04_FORECAST_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_forecast_vs_realized_SPY.png
NB04_RMSE_DELTA_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_causal_vs_baseline_rmse_delta.png
NB04_ENTROPY_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_entropy_comparison.png
NB04_CAUSAL_IMPORTANCE_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_causal_feature_importance_SPY.png
NB04_PREFIX_GAIN_FIG_PATH: /content/riskml-capstone/reports/figures/notebook04_prefix_gain_share_SPY.png
NB04_ARTIFACT_MANIFEST_PATH: /content/riskml-capstone/reports/tables/notebook04_artifact_manifest.csv


,artifact_type,artifact_path,exists_now,file_size_bytes
0,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,1055
1,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,486713
2,DATA,/content/riskml-capstone/data/processed/notebo...,True,486713
3,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,4857
4,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,245
5,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,137
6,TABLE,/content/riskml-capstone/reports/tables/causal...,True,4221
7,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,36785
8,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,1113
9,TABLE,/content/riskml-capstone/reports/tables/notebo...,True,1358


### 📌 Observations and Insights for CODE_BLOCK_N

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ VALIDATION STATUS:</strong> PASSED — All 6 figures generated and saved. Final artifact manifest shows 18/18 artifacts with exists_now = True and non-zero file sizes. Notebook 04 is complete.
</div>

<div style="border: 2px solid #5bc0de; background-color: #d9edf7; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>⚙️ WHAT THE CODE ACCOMPLISHED:</strong>

- Created the **manual DAG figure** (`notebook04_dag_figure.png`): nine rounded-box nodes (SENTIMENT, MOMENTUM, RETURNS, VALUE, VOLATILITY, RISK, ALLOCATION, MACRO, REGIME) with seven directed arrows encoding the five-edge DAG plus the two exogenous MACRO → RISK and REGIME → RISK conditioning paths.
- Created the **forecast overlay** (`notebook04_forecast_vs_realized_SPY.png`): three-model time-series comparison (EWMA, XGBoost, Causal XGBoost) against the realized 20-day forward volatility for SPY over the 500-observation test window (2023-12-28 through 2025-12-31).
- Created the **RMSE delta bar chart** (`notebook04_causal_vs_baseline_rmse_delta.png`): per-ticker RMSE deltas (causal minus XGBoost, causal minus EWMA) with a zero-line reference, sorted by XGBoost delta.
- Created the **entropy comparison bar chart** (`notebook04_entropy_comparison.png`): normalized entropy for constrained versus unconstrained models across GLD, SPY, and TLT.
- Created the **feature importance bar chart** (`notebook04_causal_feature_importance_SPY.png`): top 15 constrained features by gain for SPY.
- Created the **prefix gain share bar chart** (`notebook04_prefix_gain_share_SPY.png`): DAG-prefix-level gain comparison (constrained vs unconstrained) for SPY.
- Built the final artifact manifest with `exists_now` and `file_size_bytes` columns. All 18 artifacts confirmed present with non-zero file sizes. Saved to `notebook04_artifact_manifest.csv`.
</div>

<div style="border: 2px solid #31708f; background-color: #dce8f1; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📤 KEY OUTPUTS:</strong>

- 6 PNG figures saved to `reports/figures/` (file sizes: 72 KB to 283 KB)
- 12 CSV/data artifacts saved to `reports/tables/` and `data/processed/`
- Final manifest: `notebook04_artifact_manifest.csv` — 18 rows, 18/18 exists_now = True
- Largest figure: `notebook04_forecast_vs_realized_SPY.png` (283 KB)
- Largest table: `notebook04_causal_predictions_long.csv` (487 KB, 7,000 rows)
</div>

<div style="border: 2px solid #8a6d3b; background-color: #fcf8e3; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🔍 OBSERVATIONS:</strong>

- <span style="color: darkgreen;"><strong>✓</strong></span> **Notebook 04 is complete.** All 14 CODE_BLOCKs (A–N) executed successfully. All 18 artifacts (12 tables + 6 figures) confirmed present in the repository with non-zero file sizes.
- <span style="color: darkgreen;"><strong>✓</strong></span> The six figures are report-ready at 200 DPI with tight bounding boxes, suitable for direct insertion into the capstone Word report (Sections 4.3 and Appendix).
- <span style="color: purple;"><strong>🔗 DAG:</strong></span> The DAG figure includes the MACRO → RISK and REGIME → RISK conditioning edges alongside the five core DAG edges, providing a complete visual representation of the feature-gating architecture used in Notebook 04.
- <span style="color: darkgreen;"><strong>✓</strong></span> The forecast overlay figure for SPY shows all three models tracking the general volatility level during 2024, with a notable volatility spike around March–April 2025 where the Causal XGBoost model responds more sharply than EWMA but less aggressively than the unconstrained XGBoost.
- <span style="color: darkgreen;"><strong>✓</strong></span> The RMSE delta bar chart visually confirms that all 14 blue bars (causal minus XGBoost) are positive, while 5 orange bars (causal minus EWMA) dip below zero — consistent with the 0/14 XGBoost wins and 5/14 EWMA wins from CODE_BLOCK_G.
- <span style="color: darkgreen;"><strong>✓</strong></span> The prefix gain share chart for SPY provides the most visually striking evidence of the DAG constraint effect: VOL__ gain share jumps from 47% to 89%, while MOM__ and VAL__ bars disappear entirely from the constrained model.
- <span style="color: darkgreen;"><strong>✓</strong></span> The artifact manifest serves as a machine-readable audit trail — Notebook 06 (validation/ablation) and Notebook 07 (report figures) can reference the manifest to verify that all Notebook 04 dependencies are available before execution.

**Notebook 04 Summary of Key Findings:**
- <span style="color: darkorange;"><strong>⚠</strong></span> **H1 (Forecast Accuracy):** Not confirmed as stated. The DAG-constrained model produces higher RMSE than the unconstrained baseline for all 14 tickers (0 of 14 RMSE wins). Aggregate RMSE increases by +0.0113 (+16.2%). The constrained model does beat EWMA for 5 of 14 tickers on RMSE and shows MAE improvements for 6 of 14 tickers versus XGBoost. The result represents a meaningful accuracy-interpretability trade-off rather than a strict improvement.
- <span style="color: darkgreen;"><strong>✓</strong></span> **H4 (Interpretability):** Supported. Normalized feature-importance entropy increases for all three representative tickers (+0.003 to +0.031), indicating more balanced feature loadings under DAG constraints. The constrained model eliminates cross-node shortcuts (e.g., momentum features in the risk model) and concentrates importance on economically coherent volatility and macro features.
</div>




***

In [ ]:
import os
from getpass import getpass

os.chdir("/content/riskml-capstone")

!git config user.email "steve@youremail.com"
!git config user.name "stevearchuleta"

token = getpass("GitHub PAT: ")
!git remote set-url origin https://{token}@github.com/stevearchuleta/riskml-capstone.git

!git add -A
!git commit -m "NB04: CODE_BLOCKs A-N complete; all 18 artifacts saved, markdown observation cells added"
!git push origin main

GitHub PAT: ··········
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [27]:
# SESSION PUSH CELL — RUN AFTER ANY CODE_BLOCK THAT SAVES ARTIFACTS
# DELETE OUTPUT BEFORE COMMITTING NOTEBOOK
import os
os.chdir("/content/riskml-capstone")
!git pull origin main
!git add notebooks/04_risk_forecasting_causal.ipynb
!git add reports/tables/notebook04_*.csv
!git add reports/figures/notebook04_*.png
!git add data/processed/notebook04_*.csv
!git commit -m "fix: add XLV to ASSET_CLASS_MAP; rerun CODE_BLOCKS L-M-N to correct notebook04_asset_class_comparison.csv"
!git push origin main

From https://github.com/stevearchuleta/riskml-capstone
 * branch            main       -> FETCH_HEAD
Already up to date.
The following paths are ignored by one of your .gitignore files:
data/processed
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@cf70f1f75954.(none)')
Everything up-to-date
